# Dynamic Asset Allocation for Retirement with Age-Dependent Risk Aversion
## CME 241: Reinforcement Learning for Finance — Stage 3: Deep Q-Network (DQN)

**Stanford University · Winter 2026**

---

This notebook implements **Stage 3** of the retirement asset allocation project: a
simulation-based reinforcement learning formulation solved with a **Deep Q-Network (DQN)**.

### What this notebook covers

- A custom retirement investing environment with **continuous wealth**, **market regimes**, and **switching costs**
- Three investment strategies: VFSTX (Vanguard Short-Term Investment-Grade), VBMFX (Total Bond Market Index), VFINX (Vanguard 500 Index)
- **Age-dependent CRRA risk aversion**: γ increases linearly from age 30 to 65
- **DQN from scratch** in PyTorch: Q-network, replay buffer, target network, ε-greedy exploration
- Comparison of the learned policy against five baseline strategies
- Visualizations, interpretation, and discussion of limitations

## Table of Contents

1. [Stage 3 Objective and Relation to Phase 2](#2.-Stage-3-Objective)
2. [Modeling Assumptions](#3.-Modeling-Assumptions)
3. [Imports and Configuration](#4.-Imports-and-Configuration)
4. [Environment Design](#5.-Environment-Design)
5. [Retirement Environment Implementation](#6.-Retirement-Environment)
6. [Utility and Helper Functions](#7.-Utility-Functions)
7. [Baseline Policies](#8.-Baseline-Policies)
8. [DQN Implementation](#9.-DQN-Implementation)
9. [Training Loop](#10.-Training-Loop)
10. [Evaluation and Simulation](#11.-Evaluation)
11. [Plots and Interpretation](#12.-Plots)
12. [Results Interpretation](#13.-Results-Interpretation)
13. [Limitations](#14.-Limitations)
14. [How This Extends Phase 2 and Possible Phase 4 Improvements](#15.-Phase-4)

---
## 2. Stage 3 Objective

### Relation to Phase 2

Phase 2 solved a **discrete-state, discrete-return MDP** via exact backward induction (DP).
This gave a provably optimal policy for the gridded MDP but suffered from three key limitations:

| Phase 2 (DP) | Stage 3 (DQN) |
|---|---|
| Discrete wealth grid (60 bins) | Continuous wealth state |
| i.i.d. discrete return scenarios | Regime-dependent continuous return distributions |
| No switching costs | Configurable switching transaction costs |
| Exact DP (tractable because state space is small) | DQN learns from sampled trajectories |
| No market regime | Three regimes: bull, neutral, bear |

### Stage 3 MDP Formulation

| Component | Definition |
|-----------|-----------|
| **State** | $(t,\, W_t,\, R_t)$ — age, continuous wealth, market regime |
| **State vector** | $[\tilde{t},\, \tilde{w},\, \mathbf{1}_{\text{bull}},\, \mathbf{1}_{\text{neutral}},\, \mathbf{1}_{\text{bear}}]$ — 5-D normalized input to the Q-network |
| **Action** | 0 = VFSTX, 1 = VBMFX, 2 = VFINX |
| **Transition** | Regime evolves via Markov chain; wealth updated with regime-dependent normal return |
| **Reward** | CRRA utility of terminal wealth; small shaped intermediate reward for training stability |
| **Horizon** | $T = 35$ annual steps (ages 30–65) |
| **Risk aversion** | $\gamma(\text{age}) = \gamma_0 + k \cdot (\text{age} - 30)$, increasing with age |

---
## 3. Modeling Assumptions

### Return Model

Returns are sampled from **regime-dependent normal distributions** — a stylized approximation
calibrated to the 1990–2024 historical returns downloaded from Yahoo Finance
(VFINX: ~12.12% mean, ~17.36% std; VBMFX: ~5.17%; VFSTX: ~4.65%).  All means and standard deviations
are overwritten by the empirical calibration step below (Cell 6).

> **Important:** These are stylized simulator parameters, not precisely calibrated statistical
> models.  The goal is a rich-enough simulation for RL training, not a realistic financial model.

### Market Regimes

Three Markov regimes capture persistent market states:
- **Bull**: high equity returns, modest bond and savings rates
- **Neutral**: average conditions across all assets
- **Bear**: equity drawdown; Treasury benefits from flight-to-safety; savings rates fall

The regime is part of the state and is known to the agent (observable regime).

### Switching Costs

If the agent switches its investment choice from one year to the next, a small transaction
cost is deducted (default: 0.5% of current wealth). This encourages strategic consistency
and is more realistic than cost-free rebalancing.

### Reward Shaping

The economic objective is **CRRA utility of terminal wealth at age 65**. However, DQN with
sparse terminal rewards can be slow to learn on a 35-step horizon.  A small intermediate
shaped reward (proportional to log-wealth change, scaled by α = 0.01) is added to each step
to provide gradient signal throughout training. This shaping does not change the optimal
policy (it is potential-based in the limit) but materially accelerates convergence.

The terminal CRRA utility remains the dominant signal: the total shaped reward over an
episode is O(0.01 × 35) ≈ 0.35, while the terminal utility is O(1) or larger.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional
import warnings

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# ── Strategy index constants ──────────────────────────────────────────────
VFSTX   = 0
VBMFX   = 1
VFINX   = 2

# ── Market regime constants ───────────────────────────────────────────────
BULL    = 0
NEUTRAL = 1
BEAR    = 2

# ══════════════════════════════════════════════════════════════════════════
# CONFIG — all hyperparameters in one place; modify here to re-run
# ══════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Horizon ──────────────────────────────────────────────────────────
    start_age:      int   = 30
    retirement_age: int   = 65
    birth_year:     int   = 1960

    # ── Finances ─────────────────────────────────────────────────────────
    initial_wealth:      float = 100_000.0
    annual_contribution: float = 10_000.0
    wealth_min:          float = 1_000.0       # floor to avoid log(0)
    wealth_max:          float = 5_000_000.0   # cap for state normalization

    # ── Switching cost ────────────────────────────────────────────────────
    # Applied when the chosen action differs from the previous year.
    switching_cost_pct: float = 0.005  # 0.5% of wealth

    # ── Risk aversion (CRRA) ─────────────────────────────────────────────
    gamma_0:     float = 1.0   # risk aversion at age 30
    gamma_slope: float = 0.03  # incremental increase per year of age
    w_ref:       float = 1_000_000.0  # CRRA normalization reference ($1M target)

    # ── Intermediate reward shaping ──────────────────────────────────────
    # Scaled log-wealth change; tiny relative to terminal reward.
    # Acts as a training aid; does NOT change the optimal policy.
    intermediate_alpha: float = 0.01

    # ── Market regime transition matrix ──────────────────────────────────
    # Entry [i, j] = probability of moving from regime i to regime j.
    # Row order: [bull, neutral, bear]
    regime_transition: np.ndarray = field(default_factory=lambda: np.array([
        [0.80, 0.15, 0.05],  # bull  -> bull / neutral / bear
        [0.20, 0.60, 0.20],  # neutral -> ...
        [0.05, 0.25, 0.70],  # bear  -> ...
    ]))

    # ── Return model ─────────────────────────────────────────────────────
    # Stylized regime-dependent normal distributions (annual nominal returns).
    # Initial values are overwritten by empirical calibration from real
    # VFINX/VBMFX/VFSTX returns (1990-2024). See Cell 6.
    # Format: {action: {regime: (mean, std)}}
    return_params: Dict = field(default_factory=lambda: {
        VFSTX: {
            BULL:    (0.045, 0.005),  # High-rate environment in bull
            NEUTRAL: (0.030, 0.005),  # Mid-cycle Fed rate
            BEAR:    (0.010, 0.005),  # Low-rate recession environment
        },
        VBMFX: {
            BULL:    (0.020, 0.045),  # Modest; rates may rise in bull
            NEUTRAL: (0.040, 0.045),  # Normal yield-driven return
            BEAR:    (0.100, 0.055),  # Flight-to-safety; price appreciation
        },
        VFINX: {
            BULL:    (0.270, 0.150),  # Strong equity bull market
            NEUTRAL: (0.100, 0.150),  # Average equity year
            BEAR:    (-0.150, 0.190), # Bear market drawdown
        },
    })

    # ── DQN hyperparameters ───────────────────────────────────────────────
    n_episodes:          int   = 10000
    batch_size:          int   = 128
    replay_buffer_size:  int   = 50_000
    lr:                  float = 1e-3
    gamma_discount:      float = 0.99   # RL discount factor (not CRRA gamma)
    eps_start:           float = 1.0
    eps_end:             float = 0.05
    eps_decay:           float = 0.995  # multiplicative per-episode decay
    target_update_freq:  int   = 500    # episodes between hard target-net updates
    hidden_size:         int   = 64

    # ── Evaluation ────────────────────────────────────────────────────────
    n_eval_episodes: int = 10000

    # ── Strategy and regime names ─────────────────────────────────────────
    strategy_names: List[str] = field(default_factory=lambda: [
        "VFSTX", "VBMFX", "VFINX"
    ])
    regime_names: List[str] = field(default_factory=lambda: [
        "Bull", "Neutral", "Bear"
    ])


cfg = Config()
T_STEPS   = cfg.retirement_age - cfg.start_age  # 35
STATE_DIM = 5   # norm_age + norm_log_wealth + 3 one-hot regime

# Scheme B: 6 actions = pure picks + 50/50 blends (sensitivity analysis winner)
SCHEME_B_VECTORS = [
    (1.00, 0.00, 0.00), (0.00, 1.00, 0.00), (0.00, 0.00, 1.00),  # pure
    (0.50, 0.50, 0.00), (0.50, 0.00, 0.50), (0.00, 0.50, 0.50),  # 50/50
]
N_ACTIONS = len(SCHEME_B_VECTORS)  # 6
ACTION_LABELS = [
    'VFSTX', 'VBMFX', 'VFINX',
    '50/50 VFSTX+VBMFX', '50/50 VFSTX+VFINX', '50/50 VBMFX+VFINX',
]

print('Configuration loaded.')
print(f'  Horizon           : {T_STEPS} years (ages {cfg.start_age}-{cfg.retirement_age})')
print(f'  State dimension   : {STATE_DIM}')
print(f'  Actions           : {N_ACTIONS}  (Scheme B) {ACTION_LABELS}')
print(f'  Regimes           : {cfg.regime_names}')
print(f'  Initial wealth    : ${cfg.initial_wealth:,.0f}')
print(f'  Annual contrib.   : ${cfg.annual_contribution:,.0f}/yr')
print(f'  Switching cost    : {cfg.switching_cost_pct:.1%} of wealth if action changes')
print(f'  gamma_0           : {cfg.gamma_0}  |  gamma_slope: {cfg.gamma_slope}/yr')
print(f'  gamma at age 65   : {cfg.gamma_0 + cfg.gamma_slope * T_STEPS:.3f}')
print()
print('Return model summary (mean +/- std per regime):')
for a in range(len(cfg.strategy_names)):
    print(f'  {cfg.strategy_names[a]:<16}', end='')
    for r in range(3):
        mu, sig = cfg.return_params[a][r]
        print(f'  {cfg.regime_names[r]}: {mu:+.1%}+/-{sig:.1%}', end='')
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Historical Market Data — VFINX, VBMFX, VFSTX annual returns via yfinance
# ─────────────────────────────────────────────────────────────────────────────
# Tickers:
#   VFINX — Vanguard Total Stock Mkt Index (Inv) (equity, inception 1992)
#   VBMFX — Vanguard Total Bond Market Index      (bonds, inception 1987)
#   VFSTX — Vanguard Short-Term Investment-Grade  (savings proxy, inception 1982)
#
# Annual returns are computed from calendar-year-end adjusted close prices.
# The three series are aligned to their common overlap (1990–2024, 35 years).

try:
    import yfinance as yf
except ImportError:
    import subprocess
    subprocess.check_call([__import__('sys').executable, '-m', 'pip',
                           'install', 'yfinance', '-q'])
    import yfinance as yf

print('Downloading historical annual returns (1990–2024)…')
_dl_p3 = {}
for _ticker in ['VFINX', 'VBMFX', 'VFSTX']:
    _raw   = yf.download(_ticker, start='1989-01-01', end='2024-12-31',
                         auto_adjust=True, progress=False)
    _close = _raw['Close'].squeeze()
    _dl_p3[_ticker] = _close.resample('YE').last().pct_change().dropna()

_hist_df_p3  = pd.DataFrame(_dl_p3).dropna()
hist_years   = _hist_df_p3.index.year.tolist()
hist_vfinx_ret = _hist_df_p3['VFINX'].values  # equity  (VFINX)
hist_vbmfx_ret = _hist_df_p3['VBMFX'].values  # bonds   (VBMFX)
hist_vfstx_ret = _hist_df_p3['VFSTX'].values  # savings proxy (VFSTX)

print(f'  Aligned sample: {len(hist_years)} annual observations '
      f'({hist_years[0]}–{hist_years[-1]})')
print()
print(f"  {'Asset':<14}  {'Mean':>7}  {'Std':>7}  {'Min':>8}  {'Max':>8}")
print('  ' + '─' * 52)
for _lbl, _arr in [('VFINX (equity)',   hist_vfinx_ret),
                   ('VBMFX (bonds)',    hist_vbmfx_ret),
                   ('VFSTX (savings)',  hist_vfstx_ret)]:
    print(f'  {_lbl:<14}  {_arr.mean():>7.2%}  {_arr.std():>7.2%}'
          f'  {_arr.min():>8.2%}  {_arr.max():>8.2%}')
print()
print('  VFSTX = Vanguard Short-Term Investment-Grade Fund (savings rate proxy).')
print('  These returns are used to (1) calibrate cfg.regime_transition /  ')
print('  cfg.return_params, and (2) generate empirical bootstrap baselines.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Regime Calibration — fit Markov regime model to real VFINX return history
# ─────────────────────────────────────────────────────────────────────────────
# Market regimes are labeled using VFINX annual-return thresholds:
#   Bull   : VFINX annual return > +15 %
#   Neutral: −5 % ≤ VFINX ≤ +15 %
#   Bear   : VFINX annual return < −5 %
#
# The regime transition matrix and per-regime (μ, σ) for each asset are
# estimated from the 1990–2024 sample.  Laplace smoothing (+ 0.5 per cell)
# prevents zero-probability transitions that arise with a short history.

_vti = hist_vfinx_ret
_ief = hist_vbmfx_ret
_shy = hist_vfstx_ret

# ── Label each year's regime ─────────────────────────────────────────────
_bull_mask    = _vti >  0.15
_neutral_mask = (_vti >= -0.05) & (_vti <=  0.15)
_bear_mask    = _vti < -0.05
_regime_seq   = np.where(_bull_mask, BULL, np.where(_bear_mask, BEAR, NEUTRAL))

# ── Empirical transition matrix with Laplace smoothing ───────────────────
_trans_counts = np.zeros((3, 3))
for _t in range(len(_regime_seq) - 1):
    _trans_counts[_regime_seq[_t], _regime_seq[_t + 1]] += 1
_trans_smooth   = _trans_counts + 0.5
_emp_transition = _trans_smooth / _trans_smooth.sum(axis=1, keepdims=True)

# ── Per-regime (mean, std) for each asset ────────────────────────────────
_orig_params = {k: dict(v) for k, v in cfg.return_params.items()}
_emp_params  = {}
for _action, _arr in [(VFSTX, _shy), (VBMFX, _ief), (VFINX, _vti)]:
    _emp_params[_action] = {}
    for _regime, _mask in [(BULL, _bull_mask), (NEUTRAL, _neutral_mask), (BEAR, _bear_mask)]:
        _sub = _arr[_mask]
        if len(_sub) > 1:
            _mu  = float(_sub.mean())
            _sig = float(_sub.std(ddof=1))
        else:
            _mu  = float(_arr.mean())
            _sig = float(_arr.std(ddof=1))
        _emp_params[_action][_regime] = (_mu, _sig)

# ── Update cfg in-place (before RetirementEnv is instantiated for training) ──
cfg.regime_transition = _emp_transition
cfg.return_params     = _emp_params

# ── Summary printout ─────────────────────────────────────────────────────
_n_b, _n_n, _n_br = _bull_mask.sum(), _neutral_mask.sum(), _bear_mask.sum()
print("Regime calibration complete — cfg.return_params and cfg.regime_transition updated.")
print()
print(f"  Regime counts (1990–2024):  Bull={_n_b}, Neutral={_n_n}, Bear={_n_br}")
print()
print("  Empirical transition matrix  [from → to: Bull / Neutral / Bear]")
_prior_rows = [[0.80, 0.15, 0.05], [0.20, 0.60, 0.20], [0.05, 0.25, 0.70]]
for _i, _rname in enumerate(cfg.regime_names):
    _row = _emp_transition[_i]
    _pr  = _prior_rows[_i]
    print(f"    {_rname:8s}: [{_row[0]:.2f}, {_row[1]:.2f}, {_row[2]:.2f}]"
          f"  (was [{_pr[0]:.2f}, {_pr[1]:.2f}, {_pr[2]:.2f}])")
print()
print("  Return params per regime  (mean ± std):")
for _a in range(len(cfg.strategy_names)):
    print(f"    {cfg.strategy_names[_a]:<16}", end='')
    for _r in range(3):
        _mu, _sig = cfg.return_params[_a][_r]
        _om, _os  = _orig_params[_a][_r]
        print(f"  {cfg.regime_names[_r]}: {_mu:+.1%}±{_sig:.1%} (was {_om:+.1%}±{_os:.1%})", end='')
    print()
print()
print("  DQN training will use these empirically-calibrated parameters.")

---
## 5. Environment Design

### State Representation

The environment state at each step is encoded as a **5-dimensional normalized vector**
suitable for neural network input:

$$\mathbf{s}_t = \left[\, \underbrace{\frac{t}{35}}_{\tilde{t}},\;\underbrace{\frac{\log_{10} W_t}{\log_{10} W_{\max}}}_{\tilde{w}},\;\underbrace{\mathbf{1}_{R_t=\text{bull}},\; \mathbf{1}_{R_t=\text{neutral}},\;\mathbf{1}_{R_t=\text{bear}}}_{\text{one-hot regime}} \,\right]$$

Using log-wealth for normalization is natural since wealth grows (roughly) exponentially.
One-hot regime encoding is preferable to a single integer because it avoids imposing an
arbitrary numeric ordering on the regimes.

### Wealth Dynamics

$$W_{t+1} = \max\!\left(\,(W_t + c)\,(1 + r_t) - \text{cost}_t,\; W_{\min}\right)$$

where:
- $c$ = fixed annual contribution (\$10,000 by default)
- $r_t \sim \mathcal{N}(\mu_{a,R_t},\, \sigma_{a,R_t})$, clipped to $[-60\%, +60\%]$
- $\text{cost}_t = \lambda \cdot W_t$ if $a_t \neq a_{t-1}$, else $0$

### Regime Transitions

The market regime $R_t$ follows a Markov chain with transition matrix $P$:

$$P = \begin{pmatrix} 0.80 & 0.15 & 0.05 \\ 0.20 & 0.60 & 0.20 \\ 0.05 & 0.25 & 0.70 \end{pmatrix}$$

Bull and bear regimes are both persistent (diagonal > 0.70). Neutral is a transitional state.

In [ ]:
# ── Must define utility helpers before using them in the environment ─────
# (gamma_from_age and crra_utility are defined in the next code cell;
#  they are referenced here via forward call — fine since Python resolves
#  names at call time, not definition time.)


class RetirementEnv:
    """
    Finite-horizon retirement investing environment.

    State vector (5-D):
        [norm_age, norm_log_wealth, is_bull, is_neutral, is_bear]

    Actions (Scheme B, 6):
        0 = VFSTX  |  1 = VBMFX  |  2 = VFINX
        3 = 50/50 VFSTX+VBMFX  |  4 = 50/50 VFSTX+VFINX  |  5 = 50/50 VBMFX+VFINX

    Episode:
        Starts at age 30, ends at age 65 (35 annual steps).

    Wealth dynamics:
        w_{t+1} = max( (w_t + contribution) * (1 + return) - switch_cost, w_min )
    """

    def __init__(self, cfg: Config, seed: Optional[int] = None):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        # Internal state (set by reset())
        self.age         = cfg.start_age
        self.wealth      = cfg.initial_wealth
        self.regime      = NEUTRAL
        self.prev_action = None
        self.step_count  = 0

    def reset(self, start_regime: Optional[int] = None) -> np.ndarray:
        """Reset to age 30 and return the initial state vector."""
        self.age         = self.cfg.start_age
        self.wealth      = self.cfg.initial_wealth
        self.regime      = int(self.rng.integers(0, 3)) if start_regime is None else start_regime
        self.prev_action = None
        self.step_count  = 0
        return self._get_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool]:
        """
        Take one year-long step.

        Returns:
            next_state : np.ndarray (5-D)
            reward     : float
            done       : bool  (True at age 65)
        """
        assert 0 <= action < N_ACTIONS, f'Invalid action {action}'

        # ── Switching cost ────────────────────────────────────────────────
        if self.prev_action is not None and action != self.prev_action:
            switch_cost = self.cfg.switching_cost_pct * self.wealth
        else:
            switch_cost = 0.0

        # ── Sample annual return (blended, regime-dependent clipped normal) ──
        allocation = SCHEME_B_VECTORS[action]
        annual_return = 0.0
        for strat_idx, weight in enumerate(allocation):
            if weight > 1e-10:
                mu, sig = self.cfg.return_params[strat_idx][self.regime]
                r = float(np.clip(self.rng.normal(mu, sig), -0.60, 0.60))
                annual_return += weight * r

        # ── Wealth update ─────────────────────────────────────────────────
        prev_wealth = self.wealth
        self.wealth = (self.wealth + self.cfg.annual_contribution) * (1.0 + annual_return)
        self.wealth -= switch_cost
        self.wealth  = max(self.wealth, self.cfg.wealth_min)

        # ── Regime transition ─────────────────────────────────────────────
        self.regime = int(self.rng.choice(3, p=self.cfg.regime_transition[self.regime]))

        # ── Advance age and step counter ──────────────────────────────────
        self.prev_action = action
        self.age        += 1
        self.step_count += 1
        done = (self.age >= self.cfg.retirement_age)

        # ── Reward (Baseline) ─────────────────────────────────────────────────────
        # Step-wise log-wealth change at every step, including the terminal step.
        # No CRRA curvature; no sparse terminal signal.
        delta_log = np.log(max(self.wealth, 1.0)) - np.log(max(prev_wealth, 1.0))
        reward = self.cfg.intermediate_alpha * delta_log

        return self._get_state(), reward, done

    def _get_state(self) -> np.ndarray:
        """Build the 5-D normalized state vector."""
        norm_age   = (self.age - self.cfg.start_age) / T_STEPS
        log_w      = np.log10(max(self.wealth, self.cfg.wealth_min))
        norm_log_w = log_w / np.log10(self.cfg.wealth_max)
        regime_onehot = np.zeros(3, dtype=np.float32)
        regime_onehot[self.regime] = 1.0
        return np.array([norm_age, norm_log_w, *regime_onehot], dtype=np.float32)


class RetirementEnvFractional:
    """
    Retirement environment supporting blended/fractional strategy allocations.

    Each action index maps to an allocation vector (w_VFSTX, w_VBMFX, w_VFINX)
    summing to 1.0. The blended annual return is:
        r_blend = sum_i  w_i * N(mu_i(regime), sigma_i(regime))
    Switching cost applies when the action index (hence allocation) changes.
    State encoding is identical to RetirementEnv (5-D normalized vector).
    """

    def __init__(self, cfg: Config, action_vectors: list,
                 seed: Optional[int] = None):
        self.cfg            = cfg
        self.action_vectors = action_vectors        # list of (w0, w1, w2)
        self.n_actions      = len(action_vectors)
        self.rng            = np.random.default_rng(seed)
        self.age            = cfg.start_age
        self.wealth         = cfg.initial_wealth
        self.regime         = NEUTRAL
        self.prev_action    = None
        self.step_count     = 0

    def reset(self, start_regime: Optional[int] = None) -> np.ndarray:
        self.age         = self.cfg.start_age
        self.wealth      = self.cfg.initial_wealth
        self.regime      = (int(self.rng.integers(0, 3))
                            if start_regime is None else start_regime)
        self.prev_action = None
        self.step_count  = 0
        return self._get_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool]:
        assert 0 <= action < self.n_actions, f'Invalid action {action}'
        allocation = self.action_vectors[action]

        if self.prev_action is not None and action != self.prev_action:
            switch_cost = self.cfg.switching_cost_pct * self.wealth
        else:
            switch_cost = 0.0

        blended_return = 0.0
        for strat_idx, weight in enumerate(allocation):
            if weight > 1e-10:
                mu, sig = self.cfg.return_params[strat_idx][self.regime]
                r = float(np.clip(self.rng.normal(mu, sig), -0.60, 0.60))
                blended_return += weight * r

        prev_wealth  = self.wealth
        self.wealth  = (self.wealth + self.cfg.annual_contribution) * (1.0 + blended_return)
        self.wealth -= switch_cost
        self.wealth  = max(self.wealth, self.cfg.wealth_min)

        self.regime = int(self.rng.choice(3, p=self.cfg.regime_transition[self.regime]))

        self.prev_action = action
        self.age        += 1
        self.step_count += 1
        done = (self.age >= self.cfg.retirement_age)

        # Baseline: step-wise log-wealth change at every step, including terminal.
        delta_log = np.log(max(self.wealth, 1.0)) - np.log(max(prev_wealth, 1.0))
        reward    = self.cfg.intermediate_alpha * delta_log

        return self._get_state(), reward, done

    def _get_state(self) -> np.ndarray:
        norm_age   = (self.age - self.cfg.start_age) / T_STEPS
        log_w      = np.log10(max(self.wealth, self.cfg.wealth_min))
        norm_log_w = log_w / np.log10(self.cfg.wealth_max)
        regime_oh  = np.zeros(3, dtype=np.float32)
        regime_oh[self.regime] = 1.0
        return np.array([norm_age, norm_log_w, *regime_oh], dtype=np.float32)


# ── Sanity check ──────────────────────────────────────────────────────────
print('RetirementEnv class defined.')
print('RetirementEnvFractional class defined (Scheme B: 6 actions).')
print('  State vector shape: 5-D')
print('  [norm_age, norm_log_wealth, is_bull, is_neutral, is_bear]')

---
## 7. Utility and Helper Functions

### CRRA Utility

$$U(W;\,\gamma) = \begin{cases}
  \dfrac{(W/W_{\text{ref}})^{1-\gamma} - 1}{1-\gamma} & \gamma \neq 1 \\[8pt]
  \ln(W/W_{\text{ref}}) & \gamma = 1
\end{cases}$$

Dividing by $W_{\text{ref}} = \$1{,}000{,}000$ prevents numerical saturation: raw
$W^{1-\gamma}$ collapses to near zero for all grid points when $\gamma > 1$ and $W$ is
in the million-dollar range, making utility differences invisible to the optimizer.
Normalizing keeps the utility in a well-conditioned range.

### Age-Dependent Risk Aversion

$$\gamma(\text{age}) = \gamma_0 + k \cdot (\text{age} - 30)$$

At age 30: $\gamma = 2.0$.  At age 65: $\gamma = 2.0 + 0.03 \times 35 = 3.05$.
Higher $\gamma$ ⇒ more curvature in the utility function ⇒ more risk averse.

In [ ]:
def gamma_from_age(age: int) -> float:
    """CRRA risk aversion: increases linearly with age."""
    return cfg.gamma_0 + cfg.gamma_slope * (age - cfg.start_age)


def crra_utility(w: float, gamma: float) -> float:
    """
    Normalized CRRA utility:  U(w; gamma) = ((w/w_ref)^(1-gamma) - 1) / (1-gamma)

    Normalizing by w_ref prevents numerical saturation for large wealth values.
    If gamma == 1 (log utility):  U = log(w/w_ref).
    """
    w_norm = max(float(w), 1e-6) / cfg.w_ref
    if abs(gamma - 1.0) < 1e-9:
        return float(np.log(w_norm))
    return float((w_norm ** (1.0 - gamma) - 1.0) / (1.0 - gamma))


def terminal_utility(wealth: float) -> float:
    """Convenience wrapper: CRRA utility at retirement age (age 65)."""
    return crra_utility(wealth, 1.0)  # baseline: fixed gamma=1 (log utility) at retirement


# ── Now run the environment sanity check ──────────────────────────────────
env_test = RetirementEnv(cfg, seed=0)
s0 = env_test.reset(start_regime=BULL)
print(f'Initial state : age={env_test.age}, wealth=${env_test.wealth:,.0f},'
      f' regime={cfg.regime_names[env_test.regime]}')
print(f'State vector  : {s0}')
s1, r1, done1 = env_test.step(VFINX)
print(f'After VFINX step: age={env_test.age}, wealth=${env_test.wealth:,.0f},'
      f' reward={r1:.6f}, done={done1}')
print(f'Next state    : {s1}')
print()
print('Risk aversion gamma(age):')
for age in range(30, 66, 5):
    print(f'  age {age:2d}: gamma = {gamma_from_age(age):.3f}')
print()
print(f'CRRA utility at retirement (gamma={gamma_from_age(65):.2f}):')
for w in [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000]:
    print(f'  ${w:>9,.0f}: U = {terminal_utility(w):>8.4f}')

---
## 8. Baseline Policies

We evaluate six policies.  The first five are fixed rules; the sixth is the learned DQN policy.

| Policy | Rule |
|--------|------|
| Always VFSTX | VFSTX every year |
| Always VBMFX | VBMFX every year |
| Always VFINX | VFINX every year |
| Glide Path | VFINX (30–44) → VBMFX (45–54) → VFSTX (55–64); mimics a target-date fund |
| Random | Uniform random action each year |
| **DQN** | **Learned policy (trained below)** |

The **glide path** is the natural RL benchmark: it is a rule-based de-risking schedule
that any financial advisor would recognize, and it represents the current industry standard
for passive retirement investing (VFINX → VBMFX → VFSTX over the 35-year horizon).

In [ ]:
def policy_always_vfstx(state: np.ndarray, age: int) -> int:
    return VFSTX

def policy_always_vbmfx(state: np.ndarray, age: int) -> int:
    return VBMFX

def policy_always_vfinx(state: np.ndarray, age: int) -> int:
    return VFINX

def policy_random(state: np.ndarray, age: int) -> int:
    return np.random.randint(0, N_ACTIONS)

def policy_glide_path(state: np.ndarray, age: int) -> int:
    """
    Simple target-date fund glide path:
      ages 30-44  -> VFINX  (aggressive growth)
      ages 45-54  -> VBMFX  (de-risking)
      ages 55-64  -> VFSTX  (capital preservation)
    """
    if age < 45:
        return VFINX
    elif age < 55:
        return VBMFX
    else:
        return VFSTX


BASELINES = {
    "Always VFSTX":  policy_always_vfstx,
    "Always VBMFX":  policy_always_vbmfx,
    "Always VFINX":  policy_always_vfinx,
    "Glide Path":    policy_glide_path,
    "Random":        policy_random,
}

print('Baseline policies defined:', list(BASELINES.keys()))

---
## 9. DQN Implementation

### Why DQN?

The action space is small and discrete (3 choices), the state is low-dimensional (5-D),
and the environment is deterministic given the random seed. DQN is the natural fit:
it replaces the DP value table with a neural network, learns from sampled experience,
and handles continuous wealth without discretization.

### Architecture

```
Input (5) → Linear(64) → ReLU → Linear(64) → ReLU → Linear(3) → Q-values
```

Two hidden layers of width 64 are sufficient for a 5-D → 3-action problem.  Larger
networks do not improve results meaningfully here.

### Key DQN Ingredients

| Component | Purpose |
|-----------|---------|
| **Replay buffer** | Breaks temporal correlations between consecutive transitions |
| **Target network** | Stabilizes Q-targets by using a periodically updated copy of the Q-net |
| **ε-greedy exploration** | Starts fully random; exponentially decays toward 5% random |
| **Gradient clipping** | Prevents exploding gradients from sparse reward signal |

### DQN Target

$$y_t = r_t + \gamma \cdot \max_{a'} Q_{\theta^-}(s_{t+1}, a') \cdot (1 - \mathbf{1}_{\text{done}})$$

$$\mathcal{L}(\theta) = \mathbb{E}\left[(Q_\theta(s_t, a_t) - y_t)^2\right]$$

The target network $Q_{\theta^-}$ is updated every 500 episodes via hard copy.

In [ ]:
class QNetwork(nn.Module):
    """
    MLP approximating Q(state, action) for all actions simultaneously.

    Input  : 5-D normalized state vector
    Output : 3 Q-values (one per action)
    """
    def __init__(self, state_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ReplayBuffer:
    """
    Circular experience replay buffer.
    Stores (state, action, reward, next_state, done) tuples.
    """
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, float(done)))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states),      dtype=torch.float32),
            torch.tensor(np.array(actions),     dtype=torch.long),
            torch.tensor(np.array(rewards),     dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(np.array(dones),       dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


def select_action(state: np.ndarray, q_net: QNetwork, epsilon: float) -> int:
    """Epsilon-greedy action selection."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACTIONS)  # explore
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        return int(q_net(s).argmax(dim=1).item())  # exploit


def optimize_model(
    q_net: QNetwork,
    target_net: QNetwork,
    replay_buffer: ReplayBuffer,
    optimizer: optim.Optimizer,
    cfg: Config,
) -> Optional[float]:
    """
    One gradient update using a minibatch from the replay buffer.

    DQN target:  y = r + gamma * max_a' Q_target(s', a') * (1 - done)
    Loss       :  MSE(Q(s, a), y)

    Returns the loss value, or None if the buffer is not yet large enough.
    """
    if len(replay_buffer) < cfg.batch_size:
        return None

    states, actions, rewards, next_states, dones = replay_buffer.sample(cfg.batch_size)

    # Q-values for the actions that were actually taken
    q_values = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # Target Q-values — no gradient through target network
    with torch.no_grad():
        max_next_q = target_net(next_states).max(dim=1).values
        targets = rewards + cfg.gamma_discount * max_next_q * (1.0 - dones)

    loss = nn.MSELoss()(q_values, targets)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), max_norm=10.0)  # gradient clipping
    optimizer.step()
    return loss.item()


print('QNetwork, ReplayBuffer, select_action, optimize_model — all defined.')
# Quick architecture summary
demo_net = QNetwork(STATE_DIM, N_ACTIONS, hidden=cfg.hidden_size)
total_params = sum(p.numel() for p in demo_net.parameters())
print(f'QNetwork total parameters: {total_params:,}')

---
## 10. Training Loop

### Reward Shaping Note

The **baseline environment** (`RetirementEnvFractional`, Cell 6) uses a **pure
log-wealth reward** at every step — there is no terminal CRRA signal during baseline
training. The reward at each step is:

$$r_t = \alpha \cdot \Delta\log W_t, \quad \alpha = 0.01$$

This accumulates to roughly O(0.01 × 35) ≈ 0.35 over a full episode and acts as a dense
training signal. The absence of terminal CRRA utility is intentional for this baseline;
**Section 17 (Ablation Study)** explicitly compares this design against three CRRA-based
reward variants (sparse terminal, step-wise age-varying γ, step-wise constant γ).

We also use discount factor γ = 0.99 so that rewards from all 35 steps remain well-valued
(0.99^35 ≈ 0.70 at the terminal step).

### Training Progress

The training prints a progress summary every 500 episodes.
After training, we plot the smoothed episodic reward curve and DQN loss curve.

In [ ]:
def train_dqn(cfg: Config):
    """
    Main DQN training loop.

    Returns:
        q_net           : trained Q-network (policy network)
        episode_rewards : list of total reward per episode
        losses          : list of per-optimization-step losses
    """
    env = RetirementEnvFractional(cfg, SCHEME_B_VECTORS, seed=SEED)

    q_net      = QNetwork(STATE_DIM, N_ACTIONS, hidden=cfg.hidden_size)
    target_net = QNetwork(STATE_DIM, N_ACTIONS, hidden=cfg.hidden_size)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer     = optim.Adam(q_net.parameters(), lr=cfg.lr)
    replay_buffer = ReplayBuffer(cfg.replay_buffer_size)

    epsilon         = cfg.eps_start
    episode_rewards = []
    losses          = []

    for episode in range(cfg.n_episodes):
        state     = env.reset()
        ep_reward = 0.0
        done      = False

        while not done:
            action                        = select_action(state, q_net, epsilon)
            next_state, reward, done      = env.step(action)
            replay_buffer.push(state, action, reward, next_state, done)
            state     = next_state
            ep_reward += reward

            loss_val = optimize_model(q_net, target_net, replay_buffer, optimizer, cfg)
            if loss_val is not None:
                losses.append(loss_val)

        # Multiplicative epsilon decay after each episode
        epsilon = max(cfg.eps_end, epsilon * cfg.eps_decay)

        # Hard copy to target network periodically
        if (episode + 1) % cfg.target_update_freq == 0:
            target_net.load_state_dict(q_net.state_dict())

        episode_rewards.append(ep_reward)

        if (episode + 1) % 500 == 0:
            recent = np.mean(episode_rewards[-200:])
            print(f'  Episode {episode+1:>5}/{cfg.n_episodes} | '
                  f'eps={epsilon:.3f} | '
                  f'mean reward (last 200)={recent:.4f} | '
                  f'buffer={len(replay_buffer):,}')

    print('Training complete.')
    return q_net, episode_rewards, losses


print('Starting DQN training ...')
print(f'  Episodes       : {cfg.n_episodes:,}')
print(f'  Batch size     : {cfg.batch_size}')
print(f'  Replay capacity: {cfg.replay_buffer_size:,}')
print(f'  Learning rate  : {cfg.lr}')
print(f'  Target update  : every {cfg.target_update_freq} episodes')
print(f'  Epsilon decay  : {cfg.eps_start} -> {cfg.eps_end} (x{cfg.eps_decay}/ep)')
print()
q_net, episode_rewards, losses = train_dqn(cfg)

In [ ]:
# ── Figure 1: Training reward curve + DQN loss curve ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Smoothing helper
def smooth(arr, w=100):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

# ── Left: episodic total reward (smoothed) ────────────────────────────────
ax = axes[0]
ax.plot(episode_rewards, alpha=0.2, color='#1E88E5', linewidth=0.6)
if len(episode_rewards) >= 100:
    sm = smooth(episode_rewards, w=100)
    ax.plot(np.arange(99, len(episode_rewards)), sm,
            color='#1E88E5', linewidth=2.0, label='100-ep moving avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Total reward')
ax.set_title('Figure 1a — Training Curve: Episodic Reward', fontweight='bold')
ax.legend()

# ── Right: DQN loss ───────────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(losses, alpha=0.15, color='#E53935', linewidth=0.5)
if len(losses) >= 200:
    sm_loss = smooth(losses, w=200)
    ax2.plot(np.arange(199, len(losses)), sm_loss,
             color='#E53935', linewidth=2.0, label='200-step moving avg')
ax2.set_xlabel('Optimization step')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Figure 1b — DQN Training Loss', fontweight='bold')
ax2.legend()

plt.suptitle('Stage 3 DQN — Training Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_s3_1_training.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

---
## 11. Evaluation and Simulation

We run **10,000 episodes** for each of the six policies and collect:
- Terminal wealth distribution
- Terminal CRRA utility
- Action chosen at each age (for DQN policy analysis)
- 200 full wealth trajectories (for path plots)

All policies are evaluated on independent random environments (same seed for fairness).

In [ ]:
def run_evaluation(policy_fn, n_episodes: int, seed: int = 0) -> dict:
    """
    Evaluate a policy over n_episodes and collect statistics.

    Returns a dict with terminal wealth distribution, utility, action frequencies,
    and 200 stored wealth trajectories.
    """
    env = RetirementEnvFractional(cfg, SCHEME_B_VECTORS, seed=seed)
    terminal_wealths = []
    terminal_utils   = []
    action_by_age    = {age: [] for age in range(cfg.start_age, cfg.retirement_age)}
    wealth_paths     = []

    for ep in range(n_episodes):
        state = env.reset()
        done  = False
        path  = [env.wealth]
        while not done:
            current_age = env.age
            action      = policy_fn(state, current_age)
            action_by_age[current_age].append(action)
            state, _, done = env.step(action)
            path.append(env.wealth)
        terminal_wealths.append(env.wealth)
        terminal_utils.append(terminal_utility(env.wealth))
        if ep < 200:
            wealth_paths.append(np.array(path))

    tw = np.array(terminal_wealths)
    return {
        'terminal_wealth':  tw,
        'terminal_utility': np.array(terminal_utils),
        'action_by_age':    action_by_age,
        'wealth_paths':     wealth_paths,
        'mean_wealth':      float(np.mean(tw)),
        'median_wealth':    float(np.median(tw)),
        'std_wealth':       float(np.std(tw)),
        'p10':              float(np.percentile(tw, 10)),
        'p25':              float(np.percentile(tw, 25)),
        'p75':              float(np.percentile(tw, 75)),
        'p90':              float(np.percentile(tw, 90)),
        'mean_utility':     float(np.mean(terminal_utils)),
    }


# ── Build DQN policy callable ─────────────────────────────────────────────
def make_dqn_policy(q_net: QNetwork):
    def dqn_policy(state: np.ndarray, age: int) -> int:
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            return int(q_net(s).argmax(dim=1).item())
    return dqn_policy

dqn_policy_fn = make_dqn_policy(q_net)

ALL_POLICIES = {**BASELINES, 'DQN': dqn_policy_fn}

print(f'Evaluating {len(ALL_POLICIES)} policies x {cfg.n_eval_episodes:,} episodes each ...')
eval_results = {}
for name, fn in ALL_POLICIES.items():
    print(f'  {name} ...', end=' ', flush=True)
    eval_results[name] = run_evaluation(fn, cfg.n_eval_episodes, seed=0)
    print(f'done.  Median TW = ${eval_results[name]["median_wealth"]:,.0f}')

# ── Summary DataFrame ─────────────────────────────────────────────────────
rows = []
for name, res in eval_results.items():
    rows.append({
        'Policy':       name,
        'Mean ($)':     f'${res["mean_wealth"]:>11,.0f}',
        'Median ($)':   f'${res["median_wealth"]:>11,.0f}',
        'Std Dev ($)':  f'${res["std_wealth"]:>11,.0f}',
        'P10 ($)':      f'${res["p10"]:>11,.0f}',
        'P25 ($)':      f'${res["p25"]:>11,.0f}',
        'P75 ($)':      f'${res["p75"]:>11,.0f}',
        'P90 ($)':      f'${res["p90"]:>11,.0f}',
        'Mean Utility': f'{res["mean_utility"]:>9.4f}',
    })

df_eval = pd.DataFrame(rows)
print()
print('=' * 110)
print(f'  Evaluation Summary — {cfg.n_eval_episodes:,} Monte Carlo episodes per policy')
print('=' * 110)
print(df_eval.to_string(index=False))
print('=' * 110)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Empirical Bootstrap Baselines — evaluate using real historical returns
# ─────────────────────────────────────────────────────────────────────────────
# These three baselines bypass RetirementEnv entirely.  Each holds a single
# asset class for all 35 years; annual returns are drawn i.i.d. with
# replacement from the 1990–2024 historical record (35 observations).
# The i.i.d. bootstrap is still used for consistency; returns are drawn with
# No switching cost applies because the asset never changes.

def _run_empirical_eval(return_arr, n_episodes=10000, seed=99):
    """
    Evaluate an always-invested single-asset policy by bootstrapping
    from real annual returns.  Returns the same dict structure as
    run_evaluation() for seamless integration into eval_results and figures.
    """
    rng = np.random.default_rng(seed)
    terminal_wealths = []
    terminal_utils   = []
    wealth_paths     = []
    # action_by_age is kept empty here: the empirical bootstrap holds a single
    # asset class for all 35 years (no policy decision), so there are no
    # per-age action distributions to record.  The key is present to keep the
    # return dict structure compatible with run_evaluation(), though Figures 5–7
    # only use eval_results['DQN']['action_by_age'].
    action_by_age    = {age: [] for age in range(cfg.start_age, cfg.retirement_age)}

    for ep in range(n_episodes):
        wealth = cfg.initial_wealth
        path   = [wealth]
        for _ in range(T_STEPS):
            r      = float(rng.choice(return_arr, replace=True))
            wealth = max((wealth + cfg.annual_contribution) * (1.0 + r),
                         cfg.wealth_min)
            path.append(wealth)
        terminal_wealths.append(wealth)
        terminal_utils.append(terminal_utility(wealth))
        if ep < 200:
            wealth_paths.append(np.array(path))

    tw = np.array(terminal_wealths)
    return {
        'terminal_wealth':  tw,
        'terminal_utility': np.array(terminal_utils),
        'action_by_age':    action_by_age,
        'wealth_paths':     wealth_paths,
        'mean_wealth':      float(np.mean(tw)),
        'median_wealth':    float(np.median(tw)),
        'std_wealth':       float(np.std(tw)),
        'p10':              float(np.percentile(tw, 10)),
        'p25':              float(np.percentile(tw, 25)),
        'p75':              float(np.percentile(tw, 75)),
        'p90':              float(np.percentile(tw, 90)),
        'mean_utility':     float(np.mean(terminal_utils)),
    }


print('Running empirical bootstrap evaluations (5,000 episodes each)…')
for _label, _arr in [('VFINX (Historical)', hist_vfinx_ret),
                     ('VBMFX (Historical)', hist_vbmfx_ret),
                     ('VFSTX (Historical)', hist_vfstx_ret)]:
    print(f'  {_label} ...', end=' ', flush=True)
    eval_results[_label] = _run_empirical_eval(_arr, cfg.n_eval_episodes, seed=99)
    print(f'done.  Median TW = ${eval_results[_label]["median_wealth"]:,.0f}')

print(f'\nEmpirical evaluations complete.  Total policies: {list(eval_results.keys())}')

---
## 12. Plots and Interpretation

We produce seven figures:

1. Training curve and DQN loss *(above)*
2. Sample simulated wealth paths under DQN
3. Terminal wealth histograms — DQN vs all baselines
4. Bar chart: mean terminal utility across policies
5. Average action by age under the DQN policy
6. Action heatmap — DQN action as a function of age and initial wealth bin

In [ ]:
# ── Figure 2: Simulated wealth paths under DQN (log y-axis) ──────────────
fig, ax = plt.subplots(figsize=(12, 5))
age_axis   = np.arange(cfg.birth_year + cfg.start_age, cfg.birth_year + cfg.retirement_age + 1)
dqn_paths  = eval_results['DQN']['wealth_paths']

for path in dqn_paths:
    ax.plot(age_axis, np.array(path) / 1e6, alpha=0.07, color='#1565C0', linewidth=0.8)

# Percentile bands over all eval episodes
all_tw_arr = np.array(eval_results['DQN']['terminal_wealth'])
# Reconstruct distribution of wealth at each age via fresh evaluation
env_path = RetirementEnv(cfg, seed=1)
path_matrix = []
for ep in range(500):
    state = env_path.reset()
    done  = False
    path  = [env_path.wealth]
    while not done:
        action = dqn_policy_fn(state, env_path.age)
        state, _, done = env_path.step(action)
        path.append(env_path.wealth)
    path_matrix.append(path)
path_matrix = np.array(path_matrix)

p10  = np.percentile(path_matrix, 10, axis=0)
p25  = np.percentile(path_matrix, 25, axis=0)
p75  = np.percentile(path_matrix, 75, axis=0)
p90  = np.percentile(path_matrix, 90, axis=0)
med  = np.median(path_matrix, axis=0)

ax.fill_between(age_axis, p10/1e6, p90/1e6, alpha=0.10, color='#1565C0', label='P10-P90')
ax.fill_between(age_axis, p25/1e6, p75/1e6, alpha=0.20, color='#1565C0', label='P25-P75')
ax.plot(age_axis, med/1e6, color='#E53935', linewidth=2.2, label='Median')

ax.set_xlabel('Year')
ax.set_ylabel('Wealth ($M, log scale)')
ax.set_yscale('log')
ax.set_title('Figure 2 — Simulated Wealth Paths: DQN Policy\n(200 paths shown; shaded = P10-P90 and P25-P75 bands)',
             fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('fig_s3_2_wealth_paths.png', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── Figure 3: Terminal wealth histograms — modeled policies + empirical baselines ──
_mod_names_f3 = [k for k in eval_results if '(Historical)' not in k]   # 6 modeled
_emp_names_f3 = [k for k in eval_results if '(Historical)'     in k]   # 3 empirical
_all_names_f3 = _mod_names_f3 + _emp_names_f3

# Exact terminal wealth: apply actual sequential returns (no simulation/bootstrap)
exact_tws = {}
for _name, _ret_arr in [('VFINX (Historical)', hist_vfinx_ret),
                         ('VBMFX (Historical)', hist_vbmfx_ret),
                         ('VFSTX (Historical)', hist_vfstx_ret)]:
    _w = cfg.initial_wealth
    for _r in _ret_arr:
        _w = max((_w + cfg.annual_contribution) * (1.0 + _r), cfg.wealth_min)
    exact_tws[_name] = _w

# Extend palette to cover all 9 policies (6 modeled + 3 empirical)
palette = (['#00897B', '#1E88E5', '#E53935', '#FB8C00', '#8E24AA', '#43A047']
           + ['#7B1FA2', '#5E35B1', '#00838F'])
_hatch_f3 = [None] * len(_mod_names_f3) + ['//', '\\\\', 'xx']

LOG_LO, LOG_HI = 0.1, 200.0   # shared log-axis range ($0.1M – $200M)
bins_log = np.logspace(np.log10(LOG_LO), np.log10(LOG_HI), 61)

n_cols = 3
n_rows = (len(_all_names_f3) + n_cols - 1) // n_cols   # ⌈9/3⌉ = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 5 * n_rows), sharey=False)
axes = axes.flatten()

for i, name in enumerate(_all_names_f3):
    ax    = axes[i]
    res   = eval_results[name]
    tw_M  = np.maximum(res['terminal_wealth'] / 1e6, LOG_LO)   # keep > 0 for log scale
    med_M = res['median_wealth'] / 1e6
    h     = _hatch_f3[i]
    c     = palette[i] if i < len(palette) else '#888888'
    ax.hist(tw_M, bins=bins_log, color=c, alpha=0.75 if h else 0.82,
            edgecolor='white', linewidth=0.3, hatch=h)
    ax.axvline(med_M, color='black', linestyle='--', linewidth=1.8,
               label=f'Median ${med_M:.2f}M')
    if name in exact_tws:
        exact_M = exact_tws[name] / 1e6
        ax.axvline(exact_M, color='red', linestyle='-', linewidth=2.0,
                   label=f'Exact Actual ${exact_M:.2f}M')
    ax.set_title(name, fontsize=9, fontweight='bold',
                 color='#5E35B1' if h else 'black')
    ax.set_xscale('log')
    ax.set_xlim(LOG_LO, LOG_HI)
    ax.set_xlabel('Terminal Wealth ($M, log scale)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

for j in range(len(_all_names_f3), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    'Figure 3 — Terminal Wealth Distributions  (5,000 episodes)\n'
    'Rows 1–2: modeled policies (DQN + baselines, calibrated env)  ·  '
    'Row 3: empirical bootstrap from actual annual returns (1990–2024)',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.savefig('fig_s3_3_terminal_wealth.png', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# ── Figure 4: Mean terminal utility comparison — modeled + empirical baselines ──
mean_utils = {name: res['mean_utility'] for name, res in eval_results.items()}

_mod_names_u = [k for k in mean_utils if '(Historical)' not in k]
_emp_names_u = [k for k in mean_utils if '(Historical)'     in k]
_all_order_u = _mod_names_u + _emp_names_u

_palette_full = (['#00897B', '#1E88E5', '#E53935', '#FB8C00', '#8E24AA', '#43A047']
                 + ['#7B1FA2', '#5E35B1', '#00838F'])
_colors_u = [_palette_full[i] if i < len(_palette_full) else '#888888'
             for i in range(len(_all_order_u))]

names_u  = _all_order_u
values_u = [mean_utils[k] for k in _all_order_u]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names_u, values_u, color=_colors_u, alpha=0.88, edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=5, fontsize=9)
ax.set_xlabel(f'Mean CRRA Utility at Retirement  (gamma={gamma_from_age(65):.2f})')
ax.set_title(
    'Figure 4 — Mean Terminal Utility by Policy\n'
    '(Calibrated simulation · Empirical bootstrap from 1990–2024 actual returns)',
    fontweight='bold',
)
ax.invert_yaxis()
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')

# Dashed separator between modeled policies (top) and empirical baselines (bottom)
_n_mod_u = len(_mod_names_u)
ax.axhline(_n_mod_u - 0.5, color='#555555', linewidth=1.2, linestyle=':', alpha=0.8)

plt.tight_layout()
plt.savefig('fig_s3_4_mean_utility.png', bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

In [ ]:
# ── Figure 5: Effective allocation by age — DQN policy (Scheme B) ────────
dqn_action_by_age = eval_results['DQN']['action_by_age']
ages_range = list(range(cfg.start_age, cfg.retirement_age))
year_range = [age + cfg.birth_year for age in ages_range]

scheme_b_arr = np.array(SCHEME_B_VECTORS)  # shape (6, 3)

# Compute effective mean allocation to each of the 3 strategies at each age
alloc_by_age = np.zeros((len(ages_range), 3))
for i, age in enumerate(ages_range):
    acts = dqn_action_by_age[age]
    if len(acts) > 0:
        act_arr = np.array(acts)
        for a in range(N_ACTIONS):
            freq = np.mean(act_arr == a)
            alloc_by_age[i] += freq * scheme_b_arr[a]

fig, ax = plt.subplots(figsize=(12, 4))
colors_bar = ['#43A047', '#1E88E5', '#E53935']
bottom = np.zeros(len(ages_range))
for s, (sname, col) in enumerate(zip(cfg.strategy_names, colors_bar)):
    ax.bar(year_range, alloc_by_age[:, s], bottom=bottom,
           label=sname, color=col, alpha=0.85, width=0.9)
    bottom += alloc_by_age[:, s]

ax.set_xlabel('Year')
ax.set_ylabel('Mean portfolio weight')
ax.set_title('Figure 5 — DQN Policy: Effective Allocation by Age (Scheme B, 6 actions)',
             fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim(cfg.birth_year + cfg.start_age - 0.5, cfg.birth_year + cfg.retirement_age - 0.5)
plt.tight_layout()
plt.savefig('fig_s3_5_action_by_age.png', bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

In [ ]:
# ── Figure 6: DQN action as a function of age and wealth (scatter/heatmap) ─
# Sample many (age, wealth, action) tuples from DQN evaluation.
env_heat = RetirementEnvFractional(cfg, SCHEME_B_VECTORS, seed=3)
ages_h   = []
wealths_h = []
actions_h = []

for ep in range(2000):
    state = env_heat.reset()
    done  = False
    while not done:
        age_now    = env_heat.age
        w_now      = env_heat.wealth
        action     = dqn_policy_fn(state, age_now)
        ages_h.append(age_now)
        wealths_h.append(w_now)
        actions_h.append(action)
        state, _, done = env_heat.step(action)

ages_h    = np.array(ages_h) + cfg.birth_year
wealths_h = np.array(wealths_h)
actions_h = np.array(actions_h)

# 6-color palette for 6 Scheme B actions
colors_6 = ['#43A047', '#1E88E5', '#E53935', '#FB8C00', '#8E24AA', '#00ACC1']

fig, ax = plt.subplots(figsize=(13, 5))
for a in range(N_ACTIONS):
    mask = actions_h == a
    ax.scatter(ages_h[mask], wealths_h[mask] / 1e6,
               c=colors_6[a], alpha=0.12, s=4,
               label=ACTION_LABELS[a])

ax.set_xlabel('Year')
ax.set_ylabel('Wealth at decision point ($M)')
ax.set_yscale('log')
ax.set_title('Figure 6 — DQN Action as a Function of Age and Wealth (Scheme B)\n'
             '(2,000 simulated episodes; log-scale wealth; colors = chosen action)',
             fontweight='bold')
ax.legend(markerscale=5, loc='upper left', fontsize=8, ncol=2)
ax.set_xlim(cfg.birth_year + cfg.start_age - 0.5, cfg.birth_year + cfg.retirement_age - 0.5)
plt.tight_layout()
plt.savefig('fig_s3_6_action_heatmap.png', bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

In [ ]:
# ── Figure 7: DQN effective allocation by age, broken out by regime ──────
env_reg = RetirementEnvFractional(cfg, SCHEME_B_VECTORS, seed=5)
regime_action_age = {(r, age): [] for r in range(3) for age in range(cfg.start_age, cfg.retirement_age)}

for ep in range(3000):
    state = env_reg.reset()
    done  = False
    while not done:
        age_now    = env_reg.age
        reg_now    = env_reg.regime
        action     = dqn_policy_fn(state, age_now)
        regime_action_age[(reg_now, age_now)].append(action)
        state, _, done = env_reg.step(action)

scheme_b_arr2 = np.array(SCHEME_B_VECTORS)  # (6, 3)

fig, axes = plt.subplots(1, 3, figsize=(17, 4), sharey=True)
for r, (ax, rname) in enumerate(zip(axes, cfg.regime_names)):
    alloc_fracs = np.zeros((len(ages_range), 3))
    for i, age in enumerate(ages_range):
        acts = regime_action_age.get((r, age), [])
        if len(acts) > 0:
            act_arr = np.array(acts)
            for a in range(N_ACTIONS):
                freq = np.mean(act_arr == a)
                alloc_fracs[i] += freq * scheme_b_arr2[a]
    bottom = np.zeros(len(ages_range))
    for s, (sname, col) in enumerate(zip(cfg.strategy_names, colors_bar)):
        ax.bar(year_range, alloc_fracs[:, s], bottom=bottom,
               label=sname, color=col, alpha=0.85, width=0.9)
        bottom += alloc_fracs[:, s]
    ax.set_title(f'{rname} Regime', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(cfg.birth_year + cfg.start_age - 0.5, cfg.birth_year + cfg.retirement_age - 0.5)
    if r == 0:
        ax.set_ylabel('Mean portfolio weight')
    ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Figure 7 — DQN Effective Allocation by Age and Market Regime (Scheme B)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_s3_7_regime_action.png', bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

---
## 13. Results Interpretation

### Policy Ranking

Over a 35-year horizon, equity compounding dominates.  Policies that hold VFINX for longer
periods typically achieve higher terminal wealth and utility — consistent with the
Phase 2 DP finding. The DQN policy should learn to blend equity exposure in early years
with gradual de-risking as retirement approaches.

### Does the DQN Become More Conservative with Age?

Figure 5 shows the average action chosen by the DQN at each age.  A sensible learned
policy should show **higher VFINX usage at younger ages** and shift toward VFSTX/VBMFX
closer to retirement, consistent with increasing CRRA risk aversion γ(age).

If VFINX dominates at all ages, this indicates the agent has learned that equity compounding
is so strong in this parameterization that the optimal policy is nearly always-VFINX —
a finding consistent with the Phase 2 DP analysis.

### Regime-Dependent Behavior

Figure 7 breaks down DQN action choices by market regime.  A sensible agent should:
- Allocate **less to VFINX** in bear regimes (lower expected return, higher volatility)
- Potentially allocate **more to VBMFX** in bear regimes (flight-to-safety premium)
- Show **less differentiation** for VFSTX (which is near-constant across regimes)

If the DQN has learned to exploit regime information, behavior should differ meaningfully
across the three panels of Figure 7.

### Comparison with Glide Path

The glide path is a strong heuristic: it holds VFINX for 15 years, VBMFX for 10,
then VFSTX for 5. If the DQN closely matches the glide path, it confirms the heuristic
captures the broad structure of the optimal policy. Differences indicate regime-adaptive
or wealth-adaptive behavior that the fixed rule cannot express.

### Why Does the DQN Not Always Dominate?

Several factors limit DQN performance in this setting:

1. **Sparse rewards**: Only the terminal reward is economically meaningful.
   The shaped intermediate reward helps but does not fully solve the credit assignment problem.
2. **Long horizon**: With γ_discount^35 ≈ 0.70, even well-calibrated Q-values are
   noisy estimates of a 35-step cumulative reward.
3. **Simple return model**: If the optimal policy is nearly always-VFINX,
   DQN cannot meaningfully beat a simple baseline regardless of how well it trains.

---
## 14. Limitations

### Stylized Return Model

All return parameters are stylized approximations, not statistically fitted models.
Key simplifications:
- **Normal distribution**: Actual equity returns are negatively skewed with fat tails.
  A 60% bear-market clip is ad hoc and does not reproduce the 2008 crash distribution.
- **Regime independence from wealth**: In reality, regime-switching affects all asset
  classes simultaneously with complex cross-correlations (e.g., equity–bond flight-to-safety).
  Our model captures this partially through regime-specific parameters.
- **i.i.d. years within a regime**: Within a bull or bear regime, returns are drawn
  independently each year. In reality, bear markets have multi-year momentum.

### RL Challenges

| Challenge | Impact |
|-----------|--------|
| Sparse terminal reward | DQN needs shaped rewards; optimal shaping is non-trivial |
| 35-step horizon | Long-horizon value estimation is imprecise even with γ=0.99 |
| Non-stationary optimal policy | Risk aversion increases with age; Q-net must implicitly learn time-varying behavior |
| Single file action (not portfolio weights) | Real investors hold continuous mixed allocations; discrete actions are a coarse approximation |

### Comparison with Phase 2

Phase 2 DP was **exactly optimal** for the discrete-state MDP it solved.
Stage 3 DQN is **approximately optimal** for a richer continuous-state MDP.
The two models are not directly comparable because they use different state spaces,
return models, and objective specifications.

### No Inflation, Taxes, or Labor Income Risk

All returns are nominal. A 35-year horizon is significantly affected by inflation
(historical average: ~3%/yr). Taxes on capital gains would reduce effective returns.
Labor income uncertainty (job loss, disability) is absent from this model.

---
## 15. How This Extends Phase 2 

### How Stage 3 Extends Phase 2

| Dimension | Phase 2 (DP) | Stage 3 (DQN) |
|-----------|-------------|--------------|
| **Wealth representation** | 60 log-spaced discrete bins | Continuous; normalized log-wealth in NN input |
| **Solution method** | Exact backward induction | Approximate Q-learning from sampled episodes |
| **Return model** | 3 discrete scenarios (i.i.d.) | Regime-dependent normal distributions (Markov) |
| **Market regimes** | None | 3 Markov regimes (bull/neutral/bear) |
| **Switching costs** | None | 0.5% of wealth per action change |
| **Learning signal** | Full Bellman backup at every state | Mini-batch SGD on sampled transitions |
| **Scalability** | State space must remain small | Scales to richer state with same DQN code |

Stage 3 removes the main limitations of Phase 2 DP:
- **No wealth discretization**: continuous wealth eliminates the bin-approximation artifact
  that made VFSTX appear artificially optimal in Phase 2.
- **Regime-aware behavior**: the agent can learn to behave differently in bull vs. bear markets.
- **Switching costs**: the agent faces realistic friction for portfolio rebalancing.
- **Simulation-based**: the RL framework naturally handles more complex transition models
  without requiring a closed-form Bellman backup.

### Future Work

**Richer action space:**
- Replace discrete asset selection with a **continuous portfolio weight vector** $\mathbf{w} \in \Delta^2$.
  This allows mixed allocations (e.g., 60% VFINX + 30% VBMFX + 10% VFSTX).
  Requires an actor-critic method: DDPG, TD3, or SAC.

**Better RL algorithms:**
- **Proximal Policy Optimization (PPO)**: on-policy; more stable for sparse-reward settings.
- **Soft Actor-Critic (SAC)**: entropy-regularized; naturally encourages diversification.
- **Double DQN / Dueling DQN**: reduce overestimation bias; beneficial for long horizons.

**Richer state and transition model:**
- **Inflation**: add CPI as an exogenous state variable; use real returns.
- **Labor income uncertainty**: stochastic annual contribution (e.g., correlated with equity regime).
- **Taxes**: model capital gains tax on VBMFX and VFINX; VFSTX income is taxed as ordinary income.
- **Calibrated return processes**: fit VAR or regime-switching model to historical data
  (CRSP, Shiller dataset) rather than hand-tuned stylized parameters.

**Improved reward design:**
- **Consumption utility**: add intermediate consumption as a choice variable;
  utility over consumption stream rather than terminal wealth only.
- **Bequest motive**: add bequest weight to terminal utility.
- **Drawdown penalty**: incorporate Value-at-Risk or Conditional Value-at-Risk constraints.

**Richer bond model:**
- Replace the static Treasury return model with a **term structure model** (Vasicek, CIR)
  that explicitly models duration risk and yield curve dynamics.

---

*End of Stage 3 Notebook — CME 241 Winter 2026*

---
## 16. Sensitivity Analysis: Discrete Fractional Allocation Schemes

### Motivation

The original Phase 3 DQN used a **pure-pick action space**: each year the agent selects
*one* of the three strategies (VFSTX, VBMFX, or VFINX) to invest 100% of its wealth.
This is the simplest discrete approximation to a continuous portfolio problem, but it
prevents the agent from exploiting diversification benefits within a single time step.

This section evaluates whether **allowing fractional (blended) allocations** across
the three strategies improves utility and reduces downside risk, while keeping the
action space discrete and tractable for DQN.

### Three Action Schemes

| Scheme | # Actions | Action Space Description |
|--------|-----------|--------------------------|
| **A — Pure picks** | 3 | Each action = 100% in one strategy (original baseline) |
| **B — Binary 50/50 blends** | 6 | Three pure picks + three 50/50 two-asset blends |
| **C — 25% grid** | 15 | All allocations with 25%-increment weights over 3 strategies |

For Scheme C, the 15 actions include every $(w_0, w_1, w_2)$ triple with
$w_0 + w_1 + w_2 = 1$ and $w_i \in \{0, 0.25, 0.50, 0.75, 1\}$.

### Blended Wealth Dynamics

When the agent selects allocation vector $(w_0, w_1, w_2)$, the blended return is:

$$r_t^{\text{blend}} = w_0\, r_t^{\text{VFSTX}} + w_1\, r_t^{\text{VBMFX}} + w_2\, r_t^{\text{VFINX}}$$

Each $r_t^{i}$ is sampled independently from the regime-dependent normal distribution.
Wealth dynamics and reward shaping remain identical to the baseline formulation.
The switching cost triggers whenever the *allocation index* changes between steps.

### Experimental Setup

- Each scheme is trained from scratch for **5,000 episodes** (same budget as baseline).
- Evaluation uses **2,000 Monte Carlo episodes** per scheme.
- Seed 42 everywhere for reproducibility.
- All other hyperparameters (network architecture, replay buffer, ε-decay, γ) are unchanged.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 16a: Action Schemes + Fractional Environment
# ─────────────────────────────────────────────────────────────────────────────

# ── Define Allocation Schemes ─────────────────────────────────────────────────

def generate_allocation_schemes():
    """
    Returns an ordered dict mapping scheme label -> list of allocation vectors.

    Each allocation vector is a tuple (w_VFSTX, w_VBMFX, w_VFINX) summing to 1.0.

    Scheme A:  3 pure-pick actions  (original baseline action space)
    Scheme B:  6 actions = pure picks + all binary 50/50 blends
    Scheme C: 15 actions = all (w0,w1,w2) with 25%-increment weights
    """
    # ── Scheme A: pure picks ──────────────────────────────────────────────────
    scheme_a = [
        (1.00, 0.00, 0.00),   # 100% VFSTX
        (0.00, 1.00, 0.00),   # 100% VBMFX
        (0.00, 0.00, 1.00),   # 100% VFINX
    ]

    # ── Scheme B: pure picks + binary 50/50 blends ────────────────────────────
    scheme_b = list(scheme_a) + [
        (0.50, 0.50, 0.00),   # 50% VFSTX + 50% VBMFX
        (0.50, 0.00, 0.50),   # 50% VFSTX + 50% VFINX
        (0.00, 0.50, 0.50),   # 50% VBMFX + 50% VFINX
    ]

    # ── Scheme C: all 25%-increment weight combinations ───────────────────────
    # Enumerate all (a, b, c) with a+b+c=1 and a,b,c in {0, 0.25, 0.50, 0.75, 1.0}
    levels = [0.0, 0.25, 0.50, 0.75, 1.0]
    scheme_c = []
    for a in levels:
        for b in levels:
            c = round(1.0 - a - b, 10)
            if 0.0 <= c <= 1.0 + 1e-9:
                c = min(c, 1.0)           # clamp floating-point overshoot
                c = round(c, 4)
                scheme_c.append((a, b, c))

    return {
        'Scheme A\n(3 pure picks)':          scheme_a,
        'Scheme B\n(6: pure + 50/50)':       scheme_b,
        'Scheme C\n(15: 25% grid)':           scheme_c,
    }


SCHEMES = generate_allocation_schemes()

# Print scheme details
print('Allocation schemes defined:\n')
for name, vecs in SCHEMES.items():
    label = name.replace('\n', ' ')
    print(f'  {label}  — {len(vecs)} actions')
    for i, v in enumerate(vecs):
        print(f'    action {i:>2d}: VFSTX={v[0]:.2f}  VBMFX={v[1]:.2f}  VFINX={v[2]:.2f}')
    print()


# ── Fractional Retirement Environment (CRRA-terminal variant) ─────────────────
# This class has the same dynamics as RetirementEnvFractional (Cell 6) but uses
# a richer reward:  CRRA terminal utility on the final step + intermediate
# log-wealth shaping on all other steps.  It is used exclusively for training
# in the sensitivity analysis (Section 16) so that the action-scheme comparison
# is made with a reward that directly targets the CRRA objective.
# DO NOT rename this to RetirementEnvFractional — that would silently overwrite
# the log-wealth baseline class used by the main DQN and the ablation study.

class RetirementEnvFractionalCRRA:
    """
    Retirement environment with CRRA-terminal + intermediate log-wealth reward.

    Reward:
        r_t = alpha * delta_log_W              (intermediate steps)
        r_T = crra_utility(W_T, gamma(65))    (terminal step)

    All other dynamics (wealth, regimes, switching costs, state encoding)
    are identical to RetirementEnvFractional.
    """

    def __init__(self, cfg: Config, action_vectors: list,
                 seed: Optional[int] = None):
        self.cfg            = cfg
        self.action_vectors = action_vectors        # list of (w0, w1, w2)
        self.n_actions      = len(action_vectors)
        self.rng            = np.random.default_rng(seed)
        self.age            = cfg.start_age
        self.wealth         = cfg.initial_wealth
        self.regime         = NEUTRAL
        self.prev_action    = None
        self.step_count     = 0

    def reset(self, start_regime: Optional[int] = None) -> np.ndarray:
        self.age         = self.cfg.start_age
        self.wealth      = self.cfg.initial_wealth
        self.regime      = (int(self.rng.integers(0, 3))
                            if start_regime is None else start_regime)
        self.prev_action = None
        self.step_count  = 0
        return self._get_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool]:
        assert 0 <= action < self.n_actions, f'Invalid action {action}'
        allocation = self.action_vectors[action]

        # Switching cost when allocation index changes
        if self.prev_action is not None and action != self.prev_action:
            switch_cost = self.cfg.switching_cost_pct * self.wealth
        else:
            switch_cost = 0.0

        # Blended return: independently sample each strategy, weight by allocation
        blended_return = 0.0
        for strat_idx, weight in enumerate(allocation):
            if weight > 1e-10:
                mu, sig = self.cfg.return_params[strat_idx][self.regime]
                r = float(np.clip(self.rng.normal(mu, sig), -0.60, 0.60))
                blended_return += weight * r

        # Wealth dynamics
        prev_wealth  = self.wealth
        self.wealth  = (self.wealth + self.cfg.annual_contribution) * (1.0 + blended_return)
        self.wealth -= switch_cost
        self.wealth  = max(self.wealth, self.cfg.wealth_min)

        # Regime transition
        self.regime = int(self.rng.choice(3, p=self.cfg.regime_transition[self.regime]))

        # Advance
        self.prev_action = action
        self.age        += 1
        self.step_count += 1
        done = (self.age >= self.cfg.retirement_age)

        # Reward
        if done:
            reward = crra_utility(self.wealth, gamma_from_age(self.cfg.retirement_age))
        else:
            delta_log = np.log(max(self.wealth, 1.0)) - np.log(max(prev_wealth, 1.0))
            reward    = self.cfg.intermediate_alpha * delta_log

        return self._get_state(), reward, done

    def _get_state(self) -> np.ndarray:
        norm_age   = (self.age - self.cfg.start_age) / T_STEPS
        log_w      = np.log10(max(self.wealth, self.cfg.wealth_min))
        norm_log_w = log_w / np.log10(self.cfg.wealth_max)
        regime_oh  = np.zeros(3, dtype=np.float32)
        regime_oh[self.regime] = 1.0
        return np.array([norm_age, norm_log_w, *regime_oh], dtype=np.float32)


print('RetirementEnvFractionalCRRA defined (CRRA terminal + intermediate log-wealth reward).')
print(f'STATE_DIM = {STATE_DIM} (unchanged from baseline)')
print()

# Quick sanity check on Scheme C env
_env_c = RetirementEnvFractionalCRRA(cfg, SCHEMES['Scheme C\n(15: 25% grid)'], seed=99)
_s0 = _env_c.reset(start_regime=BULL)
_s1, _r1, _d1 = _env_c.step(7)   # action 7: some mixed allocation
print(f'Scheme C sanity: age={_env_c.age}, wealth=${_env_c.wealth:,.0f}, '
      f'reward={_r1:.5f}, done={_d1}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 16b: Modified Training + Evaluation Functions
# ─────────────────────────────────────────────────────────────────────────────

def train_dqn_scheme(cfg: Config,
                     action_vectors: list,
                     n_episodes: Optional[int] = None,
                     seed: int = SEED,
                     verbose: bool = True) -> Tuple:
    """
    Train DQN for the given fractional action scheme.

    Uses RetirementEnvFractionalCRRA (CRRA terminal + intermediate log-wealth
    reward) so the sensitivity comparison is driven by the CRRA objective.
    optimize_model() is reused unchanged (it is action-count–agnostic).

    Returns: (q_net, episode_rewards, losses)
    """
    if n_episodes is None:
        n_episodes = cfg.n_episodes

    n_actions = len(action_vectors)
    env = RetirementEnvFractionalCRRA(cfg, action_vectors, seed=seed)

    q_net      = QNetwork(STATE_DIM, n_actions, hidden=cfg.hidden_size)
    target_net = QNetwork(STATE_DIM, n_actions, hidden=cfg.hidden_size)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer     = optim.Adam(q_net.parameters(), lr=cfg.lr)
    replay_buffer = ReplayBuffer(cfg.replay_buffer_size)

    epsilon         = cfg.eps_start
    episode_rewards = []
    losses          = []

    for episode in range(n_episodes):
        state     = env.reset()
        ep_reward = 0.0
        done      = False

        while not done:
            # ε-greedy with scheme-specific n_actions
            if np.random.random() < epsilon:
                action = np.random.randint(n_actions)
            else:
                with torch.no_grad():
                    s = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                    action = int(q_net(s).argmax(dim=1).item())

            next_state, reward, done = env.step(action)
            replay_buffer.push(state, action, reward, next_state, done)
            state      = next_state
            ep_reward += reward

            loss_val = optimize_model(q_net, target_net, replay_buffer, optimizer, cfg)
            if loss_val is not None:
                losses.append(loss_val)

        epsilon = max(cfg.eps_end, epsilon * cfg.eps_decay)

        if (episode + 1) % cfg.target_update_freq == 0:
            target_net.load_state_dict(q_net.state_dict())

        episode_rewards.append(ep_reward)

        if verbose and (episode + 1) % 500 == 0:
            recent = np.mean(episode_rewards[-200:])
            print(f'    ep {episode+1:>5}/{n_episodes} | ε={epsilon:.3f} | '
                  f'mean reward (last 200)={recent:.4f}')

    return q_net, episode_rewards, losses


def evaluate_scheme(q_net: QNetwork,
                    action_vectors: list,
                    n_episodes: int = 2000,
                    seed: int = 0) -> dict:
    """
    Evaluate a trained DQN policy under the given fractional action scheme.

    Records: terminal wealth distribution, CRRA utilities, action indices by age,
    and up to 200 wealth trajectories for path-plot purposes.
    """
    env = RetirementEnvFractional(cfg, action_vectors, seed=seed)

    terminal_wealths = []
    terminal_utils   = []
    action_by_age    = {age: [] for age in range(cfg.start_age, cfg.retirement_age)}
    wealth_paths     = []

    for ep in range(n_episodes):
        state = env.reset()
        done  = False
        path  = [env.wealth]

        while not done:
            age = env.age
            with torch.no_grad():
                s      = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                action = int(q_net(s).argmax(dim=1).item())
            action_by_age[age].append(action)
            state, _, done = env.step(action)
            path.append(env.wealth)

        terminal_wealths.append(env.wealth)
        terminal_utils.append(terminal_utility(env.wealth))
        if ep < 200:
            wealth_paths.append(np.array(path))

    tw = np.array(terminal_wealths)
    return {
        'terminal_wealth':  tw,
        'terminal_utility': np.array(terminal_utils),
        'action_by_age':    action_by_age,
        'wealth_paths':     wealth_paths,
        'action_vectors':   action_vectors,
        'mean_wealth':      float(np.mean(tw)),
        'median_wealth':    float(np.median(tw)),
        'std_wealth':       float(np.std(tw)),
        'p10':              float(np.percentile(tw, 10)),
        'p25':              float(np.percentile(tw, 25)),
        'p75':              float(np.percentile(tw, 75)),
        'p90':              float(np.percentile(tw, 90)),
        'mean_utility':     float(np.mean(np.array(terminal_utils))),
    }


print('train_dqn_scheme() and evaluate_scheme() defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 16c: Run Sensitivity Experiments
# ─────────────────────────────────────────────────────────────────────────────
# Each scheme is trained from scratch for cfg.n_episodes (5,000) episodes,
# then evaluated for N_EVAL_SENSITIVITY (2,000) episodes.
# ─────────────────────────────────────────────────────────────────────────────

N_EVAL_SENSITIVITY = 2000   # evaluation episodes per scheme (medium budget)

sensitivity_results = {}    # scheme_label -> result dict

for scheme_name, action_vectors in SCHEMES.items():
    label = scheme_name.replace('\n', ' ')
    n_act = len(action_vectors)
    print(f'\n{"=" * 65}')
    print(f'  Scheme : {label}  ({n_act} actions)')
    print(f'  Training {cfg.n_episodes:,} episodes ...')

    # Reset global seeds for reproducibility
    np.random.seed(SEED)
    random.seed(SEED)
    torch.manual_seed(SEED)

    q_net_s, ep_rewards_s, losses_s = train_dqn_scheme(
        cfg, action_vectors,
        n_episodes=cfg.n_episodes,
        seed=SEED,
        verbose=True,
    )

    print(f'  Evaluating {N_EVAL_SENSITIVITY:,} episodes ...')
    res = evaluate_scheme(q_net_s, action_vectors,
                          n_episodes=N_EVAL_SENSITIVITY, seed=0)

    res['episode_rewards'] = ep_rewards_s
    res['losses']          = losses_s
    res['q_net']           = q_net_s
    sensitivity_results[label] = res

    print(f'  → Median TW = ${res["median_wealth"]:>12,.0f} | '
          f'Mean Utility = {res["mean_utility"]:.4f}')

print(f'\n{"=" * 65}')
print('All sensitivity experiments complete.')

# ── Quick summary table ───────────────────────────────────────────────────────
print(f'\n{"=" * 95}')
print('  Sensitivity Analysis — Quick Summary')
print('=' * 95)
hdr = (f'  {"Scheme":<30}  {"N Actions":>9}  '
       f'{"Mean ($)":>14}  {"Median ($)":>14}  {"Mean Utility":>13}')
print(hdr)
print('-' * 95)
for label, res in sensitivity_results.items():
    n_act = len(res['action_vectors'])
    short = label.split('(')[0].strip()
    print(f'  {short:<30}  {n_act:>9}  '
          f'${res["mean_wealth"]:>13,.0f}  ${res["median_wealth"]:>13,.0f}  '
          f'{res["mean_utility"]:>13.4f}')
print('=' * 95)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 16d: Comparative Plots
# ─────────────────────────────────────────────────────────────────────────────

_SCHEME_LABELS  = list(sensitivity_results.keys())
_SCHEME_COLORS  = ['#1E88E5', '#43A047', '#E53935']  # blue, green, red
_STRAT_NAMES    = cfg.strategy_names                  # ['VFSTX','VBMFX','VFINX']
_STRAT_COLORS   = ['#29B6F6', '#FFA726', '#66BB6A']   # per-strategy palette
_AGES           = np.arange(cfg.start_age, cfg.retirement_age + 1)  # 30..65
_AGES_STEPS     = list(range(cfg.start_age, cfg.retirement_age))     # 30..64

# ── Optional scipy KDE (graceful fallback to numpy histogram) ─────────────────
try:
    from scipy.stats import gaussian_kde as _scipy_kde
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False


def _density_plot(ax, data, color, label, cap_pct=99):
    """Plot a density curve for terminal wealth (capped at cap_pct percentile)."""
    cap = np.percentile(data, cap_pct)
    d   = data[data <= cap]
    if _HAS_SCIPY:
        kde    = _scipy_kde(d, bw_method='scott')
        x_grid = np.linspace(d.min(), d.max(), 400)
        ax.plot(x_grid / 1e6, kde(x_grid) * 1e6, color=color, linewidth=2.0,
                label=label)
    else:
        counts, edges = np.histogram(d / 1e6, bins=60, density=True)
        centers = (edges[:-1] + edges[1:]) / 2
        ax.plot(centers, counts, color=color, linewidth=2.0, label=label)
    ax.axvline(np.median(data) / 1e6, color=color, linestyle='--',
               linewidth=1.2, alpha=0.65)


# ── Figure SA: Mean Utility + Key Wealth Stats per Scheme ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# (a) Mean CRRA Utility
ax = axes[0]
mean_utils = [sensitivity_results[s]['mean_utility'] for s in _SCHEME_LABELS]
bars = ax.bar(range(len(_SCHEME_LABELS)), mean_utils,
              color=_SCHEME_COLORS, width=0.5, edgecolor='white')
ax.set_xticks(range(len(_SCHEME_LABELS)))
ax.set_xticklabels([s.replace('\n', '\n') for s in _SCHEME_LABELS], fontsize=9)
ax.set_ylabel('Mean CRRA Utility')
ax.set_title('(a) Mean CRRA Utility', fontweight='bold')
for bar, val in zip(bars, mean_utils):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002, f'{val:.4f}',
            ha='center', va='bottom', fontsize=9)

# (b) Median Terminal Wealth
ax = axes[1]
medians = [sensitivity_results[s]['median_wealth'] / 1e6 for s in _SCHEME_LABELS]
bars2 = ax.bar(range(len(_SCHEME_LABELS)), medians,
               color=_SCHEME_COLORS, width=0.5, edgecolor='white')
ax.set_xticks(range(len(_SCHEME_LABELS)))
ax.set_xticklabels([s.replace('\n', '\n') for s in _SCHEME_LABELS], fontsize=9)
ax.set_ylabel('Median Terminal Wealth ($M)')
ax.set_title('(b) Median Terminal Wealth', fontweight='bold')
for bar, val in zip(bars2, medians):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05, f'${val:.2f}M',
            ha='center', va='bottom', fontsize=9)

# (c) P10 / Median / P90 range (downside + upside)
ax = axes[2]
x_pos = np.arange(len(_SCHEME_LABELS))
p10s  = [sensitivity_results[s]['p10']  / 1e6 for s in _SCHEME_LABELS]
p90s  = [sensitivity_results[s]['p90']  / 1e6 for s in _SCHEME_LABELS]
for i, (label, p10, med, p90, col) in enumerate(
        zip(_SCHEME_LABELS, p10s, medians, p90s, _SCHEME_COLORS)):
    ax.plot([i, i], [p10, p90], color=col, linewidth=6, alpha=0.30, solid_capstyle='round')
    ax.plot(i, med, 'o', color=col, markersize=8, zorder=5)
ax.set_xticks(x_pos)
ax.set_xticklabels([s.replace('\n', '\n') for s in _SCHEME_LABELS], fontsize=9)
ax.set_ylabel('Terminal Wealth ($M)')
ax.set_title('(c) P10–Median–P90 Range', fontweight='bold')

fig.suptitle('Sensitivity Analysis — Utility & Wealth Summary by Action Scheme',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('fig_s3_sens_summary.png', dpi=110, bbox_inches='tight')
plt.show()
print('Figure SA (summary stats) saved → fig_s3_sens_summary.png')


# ── Figure SB: Terminal Wealth Distributions ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for i, (label, col) in enumerate(zip(_SCHEME_LABELS, _SCHEME_COLORS)):
    res  = sensitivity_results[label]
    tw   = res['terminal_wealth']
    short = label.split('(')[0].strip()
    _density_plot(ax, tw, col, f'{short}  (med=${np.median(tw)/1e6:.2f}M)')

ax.set_xlabel('Terminal Wealth ($M)')
ax.set_ylabel('Density')
ax.set_title('Sensitivity Analysis — Terminal Wealth Distributions\n'
             '(dashed = scheme median)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_s3_sens_wealth_dist.png', dpi=110, bbox_inches='tight')
plt.show()
print('Figure SB (wealth distributions) saved → fig_s3_sens_wealth_dist.png')


# ── Figure SC: Wealth Paths (Median + IQR + P10–P90) ─────────────────────────
fig, axes = plt.subplots(1, len(_SCHEME_LABELS),
                         figsize=(14, 5), sharey=True)
for i, (label, col) in enumerate(zip(_SCHEME_LABELS, _SCHEME_COLORS)):
    ax    = axes[i]
    paths = np.array(sensitivity_results[label]['wealth_paths'])   # (≤200, T+1)
    med   = np.median(paths, axis=0)
    p25   = np.percentile(paths, 25, axis=0)
    p75   = np.percentile(paths, 75, axis=0)
    p10   = np.percentile(paths, 10, axis=0)
    p90   = np.percentile(paths, 90, axis=0)

    ax.fill_between(_AGES, p10 / 1e6, p90 / 1e6, alpha=0.12, color=col)
    ax.fill_between(_AGES, p25 / 1e6, p75 / 1e6, alpha=0.28, color=col)
    ax.plot(_AGES, med / 1e6, color=col, linewidth=2.0, label='Median')
    ax.set_xlabel('Age')
    ax.set_title(label, fontsize=9)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

axes[0].set_ylabel('Wealth')
fig.suptitle('Sensitivity Analysis — Wealth Paths\n'
             '(median + IQR shaded + P10–P90 light band)',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('fig_s3_sens_wealth_paths.png', dpi=110, bbox_inches='tight')
plt.show()
print('Figure SC (wealth paths) saved → fig_s3_sens_wealth_paths.png')


# ── Figure SD: Effective Allocation by Age ────────────────────────────────────
# For each scheme, compute the mean portfolio weight assigned to each strategy
# at each age (averaging over evaluation episodes).

fig, axes = plt.subplots(1, len(_SCHEME_LABELS),
                         figsize=(14, 4.5), sharey=True)
for i, (label, col) in enumerate(zip(_SCHEME_LABELS, _SCHEME_COLORS)):
    ax         = axes[i]
    res        = sensitivity_results[label]
    act_vecs   = res['action_vectors']
    n_strats   = 3

    eff = np.zeros((len(_AGES_STEPS), n_strats))   # (35, 3)
    for t, age in enumerate(_AGES_STEPS):
        acts = res['action_by_age'].get(age, [])
        if not acts:
            continue
        for a in acts:
            eff[t] += np.array(act_vecs[a])
        eff[t] /= len(acts)

    bottom = np.zeros(len(_AGES_STEPS))
    for s in range(n_strats):
        ax.bar(_AGES_STEPS, eff[:, s], bottom=bottom,
               color=_STRAT_COLORS[s], label=_STRAT_NAMES[s], width=1.0)
        bottom += eff[:, s]

    ax.set_xlabel('Age')
    ax.set_title(label, fontsize=9)
    ax.set_ylim(0, 1.02)
    if i == 0:
        ax.set_ylabel('Mean Allocation Fraction')
    if i == len(_SCHEME_LABELS) - 1:
        ax.legend(loc='lower right', fontsize=8)

fig.suptitle('Sensitivity Analysis — Effective Portfolio Allocation by Age',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('fig_s3_sens_alloc_age.png', dpi=110, bbox_inches='tight')
plt.show()
print('Figure SD (allocation by age) saved → fig_s3_sens_alloc_age.png')


# ── Figure SE: Training Reward Curves ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for i, (label, col) in enumerate(zip(_SCHEME_LABELS, _SCHEME_COLORS)):
    ep_rews = sensitivity_results[label]['episode_rewards']
    short   = label.split('(')[0].strip()
    ax.plot(ep_rews, alpha=0.12, color=col, linewidth=0.5)
    if len(ep_rews) >= 100:
        sm = np.convolve(ep_rews, np.ones(100) / 100, mode='valid')
        ax.plot(np.arange(99, len(ep_rews)), sm, color=col,
                linewidth=2.0, label=f'{short} (100-ep avg)')

ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Sensitivity Analysis — Training Reward Curves', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_s3_sens_training.png', dpi=110, bbox_inches='tight')
plt.show()
print('Figure SE (training curves) saved → fig_s3_sens_training.png')


print('\nAll sensitivity figures saved.')
print('Files: fig_s3_sens_summary.png, fig_s3_sens_wealth_dist.png,')
print('       fig_s3_sens_wealth_paths.png, fig_s3_sens_alloc_age.png,')
print('       fig_s3_sens_training.png')

---
## 16e. Sensitivity Analysis — Interpretation Guide

### What each figure shows

| Figure | Description |
|--------|-------------|
| **SA** | Bar charts of (a) mean CRRA utility, (b) median terminal wealth, and (c) P10–median–P90 range per scheme. Captures both central tendency and downside/upside risk. |
| **SB** | Kernel density (or histogram) of terminal wealth for all schemes, overlaid. Dashed vertical lines = scheme medians. Shows distributional shape and tail behavior. |
| **SC** | Median wealth trajectory with interquartile (IQR) and P10–P90 shaded bands per scheme. Reveals wealth accumulation dynamics over the 35-year horizon. |
| **SD** | Stacked bar chart of **effective average allocation** to each strategy per age. For Scheme A this collapses to frequency of each pure pick; for B/C it shows the portfolio-weight mixture the DQN actually exploits. |
| **SE** | Training reward curves (raw + 100-episode moving average) for all three schemes on the same axes. Diagnoses convergence speed and stability differences. |

### Key questions to answer from the results

1. **Does a finer action grid improve performance?**  
   Compare mean CRRA utilities across Schemes A → B → C.
   If Scheme C > Scheme A, fractional blending adds value that the pure-pick DQN missed.

2. **What is the cost of the expanded action space?**  
   A larger action space (15 vs. 3) makes exploration harder and may slow convergence.
   Inspect training curves (Figure SE) for evidence of slower learning in Scheme C.

3. **Which blends does the DQN actually use?**  
   Figure SD reveals whether the DQN exploits intermediate allocations or converges to
   near-pure strategies anyway (a sign that the pure-pick approximation was already adequate).

4. **Does diversification reduce downside risk?**  
   Compare P10 (10th-percentile terminal wealth) across schemes.  
   If Scheme B/C raises P10 without sacrificing median, it represents a Pareto improvement.

5. **Is the 50/50 coarse grid (Scheme B) sufficient?**  
   If Scheme C adds little beyond Scheme B, the simpler 6-action space may be preferred
   for its lower exploration cost and faster convergence.

### Design notes

- The **switching cost** (0.5% of wealth) applies whenever the *action index* changes, even
  if two different indices happen to produce similar allocations. This is intentional: it
  mirrors real transaction costs and discourages high-frequency rebalancing.
- All three environments use **identical return calibration** (empirical Bull/Neutral/Bear
  parameters from 1990–2024 VFINX/VBMFX/VFSTX data), ensuring the only variable between
  schemes is the action-space granularity.
- **Seed 42** is fixed for every training run, enabling apples-to-apples comparison.
  The evaluation environment uses **seed 0** (independent from training).

---
## 17. Ablation Study: Reward Function Design

### Overview

The **baseline DQN** uses a uniform step-wise log-wealth reward at every step (including the terminal step):

$$r_t = \alpha \cdot \Delta \log W_t = \alpha \cdot \bigl[\log W_{t+1} - \log W_t\bigr], \quad \forall\, t$$

with $\alpha = 0.01$.  This provides dense, continuous feedback proportional to log-return, but contains **no explicit risk-aversion signal** — the agent maximises cumulative log-return.

We compare the baseline against **four ablations** that vary the reward function while holding all other design choices fixed (environment dynamics, Scheme B action space, DQN hyperparameters, random seeds):

| ID | Reward Design | Key Property |
|----|---------------|--------------|
| **Baseline** | $r_t = \alpha\,\Delta\!\log W$ at all $t$ | Dense log-return; no utility curvature |
| **Ablation B — Step-wise CRRA** | $r_t = U(W_{t+1};\,\gamma_t) - U(W_t;\,\gamma_t)$, $\;\gamma_t = 2 + 0.03\,(t-30)$ | Dense CRRA with age-varying $\gamma$ (1.0 → 3.05); log utility at age 30 |
| **Ablation C — Step-wise CRRA ($\gamma=2$)** | $r_t = U(W_{t+1};\,2) - U(W_t;\,2)$ | Dense CRRA with **constant** $\gamma=2$; isolates age-varying effect |
| **Ablation D — Step-wise CRRA ($\gamma=0$)** | $r_t = U(W_{t+1};\,0) - U(W_t;\,0)$ | Risk-neutral; reward proportional to absolute dollar gain |
| **Ablation E — Step-wise CRRA ($\gamma=3$)** | $r_t = U(W_{t+1};\,3) - U(W_t;\,3)$ | More risk-averse than C; higher curvature penalises downside |

### What each ablation tests

- **Step-wise CRRA, age-varying (B):** Does starting with log utility at age 30 and increasing risk aversion over time shape a more age-appropriate lifecycle glide path?
- **Constant $\gamma=2$ (C):** Does age-varying risk aversion in the reward matter?  Comparing B vs. C isolates the effect of the increasing $\gamma(\text{age})$ schedule.
- **Constant $\gamma=0$ (D):** Does risk-neutrality in the reward signal lead to a more aggressive (equity-heavy) policy?  The reward is proportional to absolute wealth gain with no curvature.
- **Constant $\gamma=3$ (E):** Does higher curvature drive a more conservative policy?  Comparing C ($\gamma=2$) vs. E ($\gamma=3$) isolates the effect of additional downside penalisation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 17a: Ablation Environments + Generic Training / Evaluation Functions
# ─────────────────────────────────────────────────────────────────────────────
# Four ablation environments, each overriding only the reward function.
# All other dynamics (wealth accumulation, switching costs, regime transitions,
# state representation) are inherited unchanged from RetirementEnvFractional.
# ─────────────────────────────────────────────────────────────────────────────


class RetirementEnvStepCRRA(RetirementEnvFractional):
    """
    Ablation B — Step-wise CRRA (age-varying gamma, starting at 1.0):
        At each step t:
            gamma_t = 1.0 + cfg.gamma_slope * (age_t - cfg.start_age)
                    = 1.0 at age 30, rising to 2.05 at age 65 (slope 0.03/yr)
            r_t = U(W_{t+1}; gamma_t) - U(W_t; gamma_t)
        At age 30 gamma_t == 1 exactly, so the log-utility branch fires:
            r_t = log(W_{t+1}) - log(W_t)
        For all subsequent ages the power-utility formula applies.
    """

    def step(self, action: int):
        w_before = self.wealth
        age_t    = self.age                   # pre-increment age
        next_state, _orig_reward, done = super().step(action)
        w_after  = self.wealth
        gamma_t  = 2.0 + self.cfg.gamma_slope * (age_t - self.cfg.start_age)
        w_ref    = self.cfg.w_ref
        if abs(gamma_t - 1.0) > 1e-9:
            exp      = 1.0 - gamma_t
            u_after  = (max(w_after,  self.cfg.wealth_min) / w_ref) ** exp / exp
            u_before = (max(w_before, self.cfg.wealth_min) / w_ref) ** exp / exp
            reward   = u_after - u_before
        else:
            reward = np.log(max(w_after,  self.cfg.wealth_min) /
                            max(w_before, self.cfg.wealth_min))
        return next_state, reward, done


class RetirementEnvStepCRRAConst(RetirementEnvFractional):
    """
    Ablation C — Step-wise CRRA (constant gamma = 2):
        Identical to Ablation B but uses a fixed risk-aversion of 2.0.
        Isolates the contribution of age-varying gamma on the learned policy.
    """

    GAMMA_CONST = 2.0

    def step(self, action: int):
        w_before = self.wealth
        next_state, _orig_reward, done = super().step(action)
        w_after = self.wealth
        gamma   = self.GAMMA_CONST
        w_ref   = self.cfg.w_ref
        exp     = 1.0 - gamma
        u_after  = (max(w_after,  self.cfg.wealth_min) / w_ref) ** exp / exp
        u_before = (max(w_before, self.cfg.wealth_min) / w_ref) ** exp / exp
        reward   = u_after - u_before
        return next_state, reward, done


class RetirementEnvStepCRRAGamma0(RetirementEnvFractional):
    """
    Ablation D — Step-wise CRRA (constant gamma = 0, risk-neutral):
        r_t = U(W_{t+1}; 0) - U(W_t; 0)
            = (W_{t+1} - W_t) / W_ref
        Reward is proportional to the absolute dollar gain each year.
    """

    GAMMA_CONST = 0.0

    def step(self, action: int):
        w_before = self.wealth
        next_state, _orig_reward, done = super().step(action)
        w_after = self.wealth
        gamma   = self.GAMMA_CONST
        w_ref   = self.cfg.w_ref
        exp     = 1.0 - gamma   # = 1.0
        u_after  = (max(w_after,  self.cfg.wealth_min) / w_ref) ** exp / exp
        u_before = (max(w_before, self.cfg.wealth_min) / w_ref) ** exp / exp
        reward   = u_after - u_before
        return next_state, reward, done


class RetirementEnvStepCRRAGamma3(RetirementEnvFractional):
    """
    Ablation E — Step-wise CRRA (constant gamma = 3, more risk-averse):
        r_t = U(W_{t+1}; 3) - U(W_t; 3)
        Higher curvature than Ablation C (gamma=2); penalises downside more heavily.
    """

    GAMMA_CONST = 3.0

    def step(self, action: int):
        w_before = self.wealth
        next_state, _orig_reward, done = super().step(action)
        w_after = self.wealth
        gamma   = self.GAMMA_CONST
        w_ref   = self.cfg.w_ref
        exp     = 1.0 - gamma   # = -2.0
        u_after  = (max(w_after,  self.cfg.wealth_min) / w_ref) ** exp / exp
        u_before = (max(w_before, self.cfg.wealth_min) / w_ref) ** exp / exp
        reward   = u_after - u_before
        return next_state, reward, done


def train_dqn_generic(env_class, cfg: Config, action_vectors: list,
                      n_episodes: Optional[int] = None,
                      seed: int = SEED, verbose: bool = True) -> Tuple:
    """
    Generic DQN training loop. Accepts any RetirementEnvFractional subclass.
    Returns: (q_net, episode_rewards, losses)
    """
    if n_episodes is None:
        n_episodes = cfg.n_episodes

    n_actions = len(action_vectors)
    env = env_class(cfg, action_vectors, seed=seed)

    q_net      = QNetwork(STATE_DIM, n_actions, hidden=cfg.hidden_size)
    target_net = QNetwork(STATE_DIM, n_actions, hidden=cfg.hidden_size)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer     = optim.Adam(q_net.parameters(), lr=cfg.lr)
    replay_buffer = ReplayBuffer(cfg.replay_buffer_size)

    epsilon         = cfg.eps_start
    episode_rewards = []
    losses          = []

    for episode in range(n_episodes):
        state     = env.reset()
        ep_reward = 0.0
        done      = False

        while not done:
            if np.random.random() < epsilon:
                action = np.random.randint(n_actions)
            else:
                with torch.no_grad():
                    s      = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                    action = int(q_net(s).argmax(dim=1).item())

            next_state, reward, done = env.step(action)
            replay_buffer.push(state, action, reward, next_state, done)
            state      = next_state
            ep_reward += reward

            loss_val = optimize_model(q_net, target_net, replay_buffer, optimizer, cfg)
            if loss_val is not None:
                losses.append(loss_val)

        epsilon = max(cfg.eps_end, epsilon * cfg.eps_decay)

        if (episode + 1) % cfg.target_update_freq == 0:
            target_net.load_state_dict(q_net.state_dict())

        episode_rewards.append(ep_reward)

        if verbose and (episode + 1) % 500 == 0:
            recent = np.mean(episode_rewards[-200:])
            print(f'    ep {episode+1:>5}/{n_episodes} | ε={epsilon:.3f} | '
                  f'mean reward (last 200)={recent:.4f}')

    return q_net, episode_rewards, losses


def evaluate_generic(q_net: QNetwork,
                     action_vectors: list,
                     n_episodes: int = 2000,
                     seed: int = 0) -> dict:
    """
    Evaluate any DQN using the standard (unmodified) environment.
    Post-hoc CRRA utility is computed for apples-to-apples comparison.
    """
    env = RetirementEnvFractional(cfg, action_vectors, seed=seed)

    terminal_wealths = []
    terminal_utils   = []
    action_by_age    = {age: [] for age in range(cfg.start_age, cfg.retirement_age)}
    wealth_paths     = []

    for ep in range(n_episodes):
        state = env.reset()
        done  = False
        path  = [env.wealth]

        while not done:
            age = env.age
            with torch.no_grad():
                s      = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                action = int(q_net(s).argmax(dim=1).item())
            action_by_age[age].append(action)
            state, _, done = env.step(action)
            path.append(env.wealth)

        terminal_wealths.append(env.wealth)
        terminal_utils.append(terminal_utility(env.wealth))
        if ep < 200:
            wealth_paths.append(np.array(path))

    tw = np.array(terminal_wealths)
    return {
        'terminal_wealth':  tw,
        'terminal_utility': np.array(terminal_utils),
        'action_by_age':    action_by_age,
        'wealth_paths':     wealth_paths,
        'mean_wealth':      float(np.mean(tw)),
        'median_wealth':    float(np.median(tw)),
        'std_wealth':       float(np.std(tw)),
        'p10':              float(np.percentile(tw, 10)),
        'p25':              float(np.percentile(tw, 25)),
        'p75':              float(np.percentile(tw, 75)),
        'p90':              float(np.percentile(tw, 90)),
        'mean_utility':     float(np.mean(np.array(terminal_utils))),
    }


print('Ablation environments defined:')
print('  RetirementEnvStepCRRA       — Ablation B: step-wise CRRA (age-varying gamma: 2.0 → 3.05)')
print('  RetirementEnvStepCRRAConst  — Ablation C: step-wise CRRA (gamma=2 constant)')
print('  RetirementEnvStepCRRAGamma0 — Ablation D: step-wise CRRA (gamma=0, risk-neutral)')
print('  RetirementEnvStepCRRAGamma3 — Ablation E: step-wise CRRA (gamma=3, more risk-averse)')
print('train_dqn_generic(), evaluate_generic() — defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 17b: Train All Four Ablation DQNs
# ─────────────────────────────────────────────────────────────────────────────
# All four use Scheme B (6 fractional actions), the same DQN hyperparameters,
# and seed=42 for training — the ONLY difference is the reward function.
# ─────────────────────────────────────────────────────────────────────────────

_ABLATIONS = [
    ('Ablation B — Step-wise CRRA',       RetirementEnvStepCRRA),
    ('Ablation C — Step-wise CRRA (γ=2)', RetirementEnvStepCRRAConst),
    ('Ablation D — Step-wise CRRA (γ=0)', RetirementEnvStepCRRAGamma0),
    ('Ablation E — Step-wise CRRA (γ=3)', RetirementEnvStepCRRAGamma3),
]

abl_nets   = {}   # label -> q_net
abl_rews   = {}   # label -> episode_rewards list
abl_losses = {}   # label -> losses list

for label, env_class in _ABLATIONS:
    print(f'\nTraining {label} ...')
    print(f'  Action space : Scheme B ({len(SCHEME_B_VECTORS)} fractional actions)')
    print(f'  Episodes     : {cfg.n_episodes:,}')
    q, rew, loss = train_dqn_generic(
        env_class, cfg, SCHEME_B_VECTORS, seed=SEED, verbose=True,
    )
    abl_nets[label]   = q
    abl_rews[label]   = rew
    abl_losses[label] = loss
    print(f'  Done.  Final 200-ep mean reward = {np.mean(rew[-200:]):.4f}')

print('\nAll four ablation DQNs trained.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 17c: Evaluate All Four Ablations + Summary Table
# ─────────────────────────────────────────────────────────────────────────────
# Evaluation uses the STANDARD (RetirementEnvFractional) environment so that
# all post-hoc CRRA utility numbers are computed from identical dynamics.
# ─────────────────────────────────────────────────────────────────────────────

N_EVAL = cfg.n_eval_episodes   # 5,000 — same budget as baseline

abl_evals = {}   # label -> eval dict

for label in abl_nets:
    print(f'Evaluating {label} over {N_EVAL:,} episodes (standard env, seed=0) ...')
    res = evaluate_generic(abl_nets[label], SCHEME_B_VECTORS, n_episodes=N_EVAL, seed=0)
    abl_evals[label] = res
    print(f'  Median wealth  : ${res["median_wealth"]:,.0f}')
    print(f'  Mean CRRA util : {res["mean_utility"]:.4f}')

# Baseline results already stored from Section 11
base_eval = eval_results['DQN']

# ── Summary table ──────────────────────────────────────────────────────────────
rows = []
all_results = [('Baseline (Log-Wealth)', base_eval)] + \
              [(label, abl_evals[label]) for label in abl_nets]

for label, res in all_results:
    rows.append({
        'Policy':            label,
        'Mean Wealth':       f'${res["mean_wealth"]:>12,.0f}',
        'Median Wealth':     f'${res["median_wealth"]:>12,.0f}',
        'Std Dev':           f'${res["std_wealth"]:>12,.0f}',
        'P10':               f'${res["p10"]:>12,.0f}',
        'P90':               f'${res["p90"]:>12,.0f}',
        'Mean CRRA Utility': f'{res["mean_utility"]:>9.4f}',
    })

df_abl = pd.DataFrame(rows).set_index('Policy')
print('\n' + df_abl.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 17d: Five-Way Ablation Comparison — Four-Panel Figure
# ─────────────────────────────────────────────────────────────────────────────

BASE_COL = '#1E88E5'   # blue   — baseline (log-wealth)
COL_B    = '#43A047'   # green  — Ablation B: step-wise CRRA (age-varying, γ: 1.0→2.05)
COL_C    = '#8E24AA'   # purple — Ablation C: step-wise CRRA (γ=2 constant)
COL_D    = '#FB8C00'   # orange — Ablation D: step-wise CRRA (γ=0, risk-neutral)
COL_E    = '#6D4C41'   # brown  — Ablation E: step-wise CRRA (γ=3, more risk-averse)

_ALL_ENTRIES = [
    ('Baseline (Log-Wealth)',
     base_eval, episode_rewards, BASE_COL),
    ('Ablation B — Step-wise CRRA',
     abl_evals['Ablation B — Step-wise CRRA'],
     abl_rews['Ablation B — Step-wise CRRA'], COL_B),
    ('Ablation C — Step-wise CRRA (γ=2)',
     abl_evals['Ablation C — Step-wise CRRA (γ=2)'],
     abl_rews['Ablation C — Step-wise CRRA (γ=2)'], COL_C),
    ('Ablation D — Step-wise CRRA (γ=0)',
     abl_evals['Ablation D — Step-wise CRRA (γ=0)'],
     abl_rews['Ablation D — Step-wise CRRA (γ=0)'], COL_D),
    ('Ablation E — Step-wise CRRA (γ=3)',
     abl_evals['Ablation E — Step-wise CRRA (γ=3)'],
     abl_rews['Ablation E — Step-wise CRRA (γ=3)'], COL_E),
]

scheme_b_arr = np.array(SCHEME_B_VECTORS)
ages_range   = list(range(cfg.start_age, cfg.retirement_age))
year_range   = [age + cfg.birth_year for age in ages_range]


def _compute_alloc_abl(action_by_age_dict, action_vectors, n_act):
    alloc = np.zeros((len(ages_range), 3))
    for i, age in enumerate(ages_range):
        acts = action_by_age_dict.get(age, [])
        if len(acts) > 0:
            act_arr = np.array(acts)
            for a in range(n_act):
                alloc[i] += np.mean(act_arr == a) * np.array(action_vectors[a])
    return alloc


fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(
    'Ablation Study — Reward Function Design: Five-Way Comparison\n'
    'Scheme B · 5,000 training episodes · 5,000 eval episodes · Seed 42 / 0',
    fontweight='bold', fontsize=12,
)

# ── Panel A: Training reward curves ──────────────────────────────────────────
ax = axes[0, 0]
for label, res, ep_rews, col in _ALL_ENTRIES:
    raw = np.array(ep_rews)
    ax.plot(raw, alpha=0.08, color=col, linewidth=0.5)
    if len(raw) >= 100:
        sm = np.convolve(raw, np.ones(100) / 100, mode='valid')
        ax.plot(np.arange(99, len(raw)), sm, color=col, linewidth=2.0,
                label=f'{label.split("—")[-1].strip()} (100-ep avg)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Episode Reward')
ax.set_title('Panel A — Training Reward Curves', fontweight='bold')
ax.legend(fontsize=7.5)
ax.text(0.97, 0.04, 'Note: reward scales differ across designs',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=7.5, style='italic', color='gray')

# ── Panel B: Terminal wealth distributions (log scale) ───────────────────────
ax = axes[0, 1]
LOG_LO, LOG_HI = 0.05, 200.0
bins_log = np.logspace(np.log10(LOG_LO), np.log10(LOG_HI), 60)
for label, res, ep_rews, col in _ALL_ENTRIES:
    tw  = res['terminal_wealth']
    lbl = label.split('—')[-1].strip()
    ax.hist(tw / 1e6, bins=bins_log, alpha=0.35, color=col, label=lbl)
    ax.axvline(np.median(tw) / 1e6, color=col, linestyle='--', linewidth=1.6)
ax.set_xscale('log')
ax.set_xlabel('Terminal Wealth ($M, log scale)')
ax.set_ylabel('Count')
ax.set_title('Panel B — Terminal Wealth Distributions', fontweight='bold')
ax.legend(fontsize=7.5)

# ── Panel C: Post-hoc CRRA utility box plots ─────────────────────────────────
ax = axes[1, 0]
util_arrays = [res['terminal_utility'] for _, res, _, _ in _ALL_ENTRIES]
labels_bp   = ['Baseline\n(Log-W)', 'Abl. B\n(γ:1→2.05)', 'Abl. C\n(γ=2)',
               'Abl. D\n(γ=0)', 'Abl. E\n(γ=3)']
colors_bp   = [BASE_COL, COL_B, COL_C, COL_D, COL_E]

bp = ax.boxplot(
    util_arrays,
    labels=labels_bp,
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2.5),
    widths=0.45,
    showfliers=False,
)
for patch, col in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)

ax.set_ylabel(f'Post-hoc CRRA Utility  (γ = {gamma_from_age(cfg.retirement_age):.2f} at retirement)')
ax.set_title('Panel C — Effective CRRA Utility at Retirement\n(post-hoc, standard eval env)',
             fontweight='bold')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
for x, arr, col in zip(range(1, 6), util_arrays, colors_bp):
    ax.text(x + 0.27, np.mean(arr), f'mean={np.mean(arr):.3f}',
            va='center', fontsize=8.0, color=col)

# ── Panel D: VFINX equity allocation by age — all five agents ────────────────
ax = axes[1, 1]
ls_styles = ['-', '--', ':', '-.', '-']
for (label, res, ep_rews, col), ls in zip(_ALL_ENTRIES, ls_styles):
    alloc = _compute_alloc_abl(res['action_by_age'], scheme_b_arr, N_ACTIONS)
    lbl   = label.split('—')[-1].strip()
    ax.plot(year_range, alloc[:, 2], color=col, linewidth=2.0,
            linestyle=ls, label=f'{lbl} — VFINX')
ax.set_xlabel('Year')
ax.set_ylabel('Mean portfolio weight in VFINX (equity)')
ax.set_ylim(-0.05, 1.05)
ax.set_title('Panel D — Equity (VFINX) Allocation by Age', fontweight='bold')
ax.legend(fontsize=7.5)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('fig_s3_ablation_comparison.png', dpi=110, bbox_inches='tight')
plt.show()
print('Ablation comparison figure saved → fig_s3_ablation_comparison.png')

---
## 17e. Ablation Study — Interpretation Guide

### What each panel shows

| Panel | Description |
|-------|-------------|
| **A — Training curves** | Episode reward (raw + 100-ep moving average) for all five agents. Reward scales differ: log-wealth baseline ≈ O(0.01–0.05); step-wise CRRA ≈ O(−0.1 to 0). Compare *convergence speed and stability*, not absolute levels. |
| **B — Wealth distributions** | Overlaid histograms (log scale) of terminal wealth across 5,000 evaluation episodes. Dashed verticals are medians. Shows whether reward design affects wealth accumulation or primarily risk-adjusted utility. |
| **C — Effective CRRA Utility** | Post-hoc CRRA utility (γ = 3.05 at age 65) computed from terminal wealth in the **standard evaluation environment** for all five agents. The true economic objective — directly comparable regardless of what was optimised during training. |
| **D — Equity allocation by age** | Mean VFINX weight per year. Ablations with higher constant γ should produce a more conservative glide path; Ablation B's increasing γ should show a more pronounced de-risking tilt near retirement. |

### Key questions answered

1. **Does the baseline's log-wealth reward produce a sensible policy?**
   If Panel C shows competitive CRRA utility, the simpler reward is nearly sufficient and the CRRA ablations add limited value.

2. **Does age-varying γ (Ablation B, γ: 1→2.05) beat a fixed γ (Ablations C/D/E)?**
   If Panel D shows B de-risking more aggressively near retirement, the increasing γ(age) is actively shaping the lifecycle glide path — not just rescaling rewards.

3. **Does constant γ monotonically shape the glide path? (D γ=0 vs. C γ=2 vs. E γ=3)**
   Higher γ penalises downside wealth states more heavily; Panel D should show progressively more conservative equity allocation as γ rises from 0 → 2 → 3.

4. **Which reward achieves the best risk-adjusted outcome?**
   Panel C is the definitive comparison; the agent achieving the highest mean/median post-hoc CRRA utility has learned the most economically valuable policy.

### Design notes

- All five agents use **Scheme B** (6 fractional actions), `SEED = 42` for training, and seed `0` for evaluation with `5,000` episodes each.
- Evaluation always uses **`RetirementEnvFractional`** (unmodified baseline dynamics), ensuring stochastic paths are identical across all agents.
- The intermediate `α × Δlog W` scaling applies only to the baseline; ablation environments override the reward entirely.
- Ablation B uses `gamma_t = 1.0 + cfg.gamma_slope × (age − 30)`, giving γ: 1.0 at age 30 → 2.05 at age 65 (log utility at the first step, power utility thereafter).
- Ablations C, D, E use fixed γ = 2, 0, 3 respectively, isolating the effect of risk-aversion curvature independently of age.